# Bengali Hallucination Detection -- Gemma-3-12B + RAG (Dual-GPU)

Classifies a Bengali (Question, Response) pair as **faithful (1)** or **hallucinated (0)**. A
single, self-contained kernel that reproduces predictions from the raw test set end to end,
fully offline, within Kaggle's code-competition limits (2xT4 GPU, <9h runtime, <50GB weights). For meaning/idiom and math-reasoning questions, the LLM's prompt also includes the row's own rule-based categorization (domain, subdomain, task type, complexity) as extra context before it judges.

## Before you run -- attach these Kaggle inputs

**Datasets** (Add Input -> Datasets):
- [bangla-knowledgebase](https://www.kaggle.com/datasets/shahriar26s/bangla-knowledgebase) -- factual Q&A, grammar, and idiom reference KB
- [bnwiki-rag](https://www.kaggle.com/datasets/shahriar26s/bnwiki-rag) -- Bengali Wikipedia + law RAG corpus (embeddings/chunks)
- [spelling-checker-v1](https://www.kaggle.com/datasets/mahadivai/spelling-checker-v1) -- Bangla spelling dictionary (optional; the spelling-question override no-ops if absent)

**Models** (Add Input -> Models -- match `RAG_EMBED_PATH`/`RAG_RERANK_PATH`/`MODEL_NAME` in the config cell):
- [BAAI/bge-m3](https://www.kaggle.com/models/andreasbis/baai-bge-m3/Transformers/default/1) -- dense retriever for RAG
- [BAAI/bge-reranker-v2-m3](https://www.kaggle.com/models/andreasbis/baai-bge-reranker-v2-m3/Transformers/default/1) -- reranker for RAG
- `unsloth/gemma-3-12b-it-bnb-4bit` -- main inference model (linked from Hugging Face; search Kaggle Models for this name)

**Phase-2 (held-out fold):** in section **1. Environment Setup**, the first code cell, find the
line `CSV_PATH = "..."` (right next to `MODEL_NAME`/`GOLD_KB_PATH`) and point it to the held-out
test file's path. Nothing else needs to change -- the pipeline adapts to any row count/order
automatically.

**Reading the progress:** the launch cell polls both GPU workers and prints a status line like
this every cycle:

```
  [ X.X min] gpu0 (running) n/total:  fast .../...  mean .../...  reas .../...
             gpu1 (running) n/total:  fast .../...  mean .../...  reas .../...
```

**This is healthy** -- both GPUs show `(running)` with counts climbing (even slowly); progress is
checkpointed per tier and per GPU, so just let it continue. **It's a problem** if a worker's line
instead shows it has already **exited** while every count for that GPU is still 0 (e.g.
`gpu1 (rc=0) 0/total:  fast 0/...  mean 0/...  reas 0/...`) -- that worker never actually started
processing. In that case, restart the notebook fully from the first cell rather than just
re-running.


Pipeline (no separate files, no chained notebooks):
1. **Load raw CSV** (`id, context, prompt_bn, response_bn`).
2. **Rule-based categorization in-memory** -- adds `domain / subdomain / task_type /
   complexity_level / has_context / is_mcq / has_negation / has_number_or_date / language_mix`
   (pure regex, no GPU).
3. **Knowledge-base match** -- offline reference KBs:
   - A Bengali Q->A reference of common factual questions (compiled from public Bengali sources).
     When a question closely matches an entry, the reference answer is passed to the LLM as
     evidence to help it judge -- the LLM still decides.
   - A Bengali idiom (বাগধারা) reference mapping idioms to their figurative meaning (from a public
     Bengali idiom dataset); for idiom/word-meaning questions the meaning is passed as a hint and
     the LLM still judges (paraphrase-tolerant).
   - **RAG (optional, auto-detected)** -- for NO-CONTEXT questions, retrieve from a prebuilt
     Bengali-Wikipedia + bdlaws corpus (BGE-M3 dense search) and **rerank with
     bge-reranker-v2-m3**, injecting only genuinely-relevant passages as advisory evidence. The
     reranker is what prevents noise. Skipped automatically if the RAG dataset isn't attached.
4. **Verdict (speed-first, LLM tiers):** every row is judged by the LLM; the tier only sets the
   prompt style and token budget.
   - **Reasoning-Calculation**: a trimmed `<think>` step for step-by-step math.
   - **Idiom/word-meaning** (বাগধারা, শাব্দিক অর্থ, ভাবার্থ): a brief "state the figurative
     meaning, then judge" step -- a plain one-line verdict under-performs on idioms.
   - **Everything else**: a short greedy verdict. Context questions are checked strictly
     against the Context.
5. **Write `submission.csv`** (`id,label`).

> **Why this is fast enough for ~5000 rows in <9h:** thinking/reasoning-heavy tiers only run on
> the small slices that actually need them (math, idioms); the bulk gets a ~1-line non-thinking
> verdict on a trimmed prompt.

> **Dual-GPU:** data-parallels across both Kaggle T4s -- two independent model copies, one per
> physical GPU, each processing half the test set via a standalone worker script
> (`dual_gpu_worker.py`, written to disk by this notebook). Falls back to a single-GPU path
> automatically if only one GPU is visible.

## 1. Environment Setup

In [ ]:
!pip install -q unsloth
!pip install -q scikit-learn matplotlib seaborn pandas numpy

In [ ]:
import os, warnings, logging as _logging, contextlib, io
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"]    = "0,1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Force-disable HF's Xet storage backend BEFORE unsloth/transformers ever import -- the crash was
# HF's Xet CDN returning "403 / SignatureError: invalid key pair id" on model download. unsloth's
# own automatic retry-after-failure (HF_HUB_DISABLE_XET=1) happened too late (after wasting time on
# the broken path) and still got stuck. Setting this up front skips Xet entirely from the first
# request instead of failing into it.
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Quiet the import-time chatter (unsloth 🦥 banner + transformers deprecation-alias lines).
# These print DURING import, so env/warnings must be set BEFORE importing, and the imports are
# wrapped in a stdout/stderr/warnings silencer. Import exceptions still propagate normally.
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
warnings.filterwarnings("ignore")
for _n in ("transformers", "unsloth", "unsloth_zoo", "accelerate", "bitsandbytes", "huggingface_hub"):
    _logging.getLogger(_n).setLevel(_logging.ERROR)

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()), \
        warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import unsloth
    from unsloth import FastLanguageModel
    import torch
    from transformers import StoppingCriteria, StoppingCriteriaList
    try:
        from transformers.utils import logging as _hf_logging
        _hf_logging.set_verbosity_error()
    except Exception:
        pass

import re, gc, json, time, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer

DEVICE             = "cuda" if torch.cuda.is_available() else "cpu"
MAX_SEQ_LENGTH     = 2304
MAX_INPUT_LEN      = 1792
BATCH_SIZE         = 10
BATCH_SIZE_MEANING = 7
BATCH_SIZE_MATH    = 3
SEQ_TOKEN_BUDGET   = 6500
SEED               = 3407
torch.manual_seed(SEED)

# HF Hub Xet-storage was 403-ing unauthenticated downloads (crashed both dual-GPU workers on
# "SignatureError: invalid key pair id"). HF_HUB_DISABLE_XET (set above) skips that path. This
# also tries an HF_TOKEN Kaggle secret if present (authenticated downloads get a healthier rate
# limit) -- harmless no-op if you don't have one.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets (authenticated HF downloads enabled).")
except Exception:
    pass

MODEL_NAME = "unsloth/gemma-3-12b-it-bnb-4bit"

CSV_PATH      = "/kaggle/input/competitions/bengali-hallucination/test set.csv"
GOLD_KB_PATH  = "/kaggle/input/datasets/shahriar26s/bangla-knowledgebase/bangla_factual_qa_kb.csv"
IDIOM_KB_PATH = "/kaggle/input/datasets/shahriar26s/bangla-knowledgebase/idiom_kb.csv"

USE_GOLD_HINT = True
GOLD_SIM_TH   = 0.90
USE_IDIOM_HINT = True
USE_GRAMMAR_HINT = True
GRAMMAR_KB_PATH = "/kaggle/input/datasets/shahriar26s/bangla-knowledgebase/grammar_kb.csv"
GRAMMAR_SIM_TH  = 0.70

# Bangla spelling dictionary (optional -- deterministic spelling-question override).
# Glob patterns so it is found wherever the Kaggle dataset is attached; override no-ops if absent.
USE_SPELL_DICT   = True
SPELL_DICT_GLOBS = ["/kaggle/input/datasets/mahadivai/spelling-checker-v1/right_file.txt",
                    "/kaggle/input/datasets/mahadivai/spelling-checker-v1/bangla_word_huge_dataset.csv",
                    "/kaggle/input/**/right_file.txt",
                    "/kaggle/input/**/bangla_word_huge_dataset.csv"]

CALIBRATION = "strict"

USE_RAG        = True
RAG_EMBED_PATH  = "/kaggle/input/models/andreasbis/baai-bge-m3/transformers/default/1"
RAG_RERANK_PATH = "/kaggle/input/models/andreasbis/baai-bge-reranker-v2-m3/transformers/default/1"
RAG_EMB_NPY     = "/kaggle/input/datasets/shahriar26s/bnwiki-rag/embeddings.npy"
RAG_CHUNKS      = "/kaggle/input/datasets/shahriar26s/bnwiki-rag/chunks.parquet"
RAG_N_CAND      = 20
RAG_TOPK        = 2
RAG_RERANK_MIN  = 0.0
RAG_MAX_CHARS   = 700

REASONING_KINDS = {"reasoning"}
GEN_CONFIG = {
    "fast":      dict(max_new_tokens=110, do_sample=False),
    "meaning":   dict(max_new_tokens=150, do_sample=False),
    "reasoning": dict(max_new_tokens=418, do_sample=True, temperature=0.6, top_p=0.95),
}

print(f"unsloth {unsloth.__version__} | torch {torch.__version__} | device {DEVICE}")
if DEVICE == "cuda":
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} ({p.total_memory/1e9:.1f} GB)")

## 2. Load Raw Data

Reads the raw CSV. If a `label` column happens to be present, it is dropped before the model
sees any of the data.

In [ ]:
print("Loading:", CSV_PATH)
df = pd.read_csv(CSV_PATH)
if "id" not in df.columns:
    df.insert(0, "id", range(1, len(df) + 1))

df_test = df.drop(columns=["label"]) if "label" in df.columns else df.copy()

print(f"Rows: {len(df_test)}")
print(df_test.head(2))

## 3. Rule-Based Categorization (in-memory, no GPU)

Same validated regex/keyword rules as the standalone categorizer, applied directly to the
loaded dataframe — no intermediate file. Produces the columns the prompt builder consumes.

In [ ]:
DOMAIN_TOPICS = {
    "History_Politics": {
        "keywords": ["মুক্তিযুদ্ধ", "স্বাধীনতা", "বঙ্গবন্ধু", "শেখ মুজিব", "পাকিস্তান",
                     "মুঘল", "সম্রাট", "সুলতান", "রাজা", "রাজত্ব", "বিপ্লব", "আন্দোলন",
                     "সংবিধান", "রাষ্ট্রপতি", "প্রধানমন্ত্রী", "মন্ত্রী", "সংসদ",
                     "জাতিসংঘ", "চুক্তি", "নির্বাচন", "সাম্রাজ্য", "ঔপনিবেশিক"],
        "topics": {
            "Liberation War"        : ["মুক্তিযুদ্ধ", "মুক্তিযোদ্ধা", "১৯৭১", "স্বাধীনতা যুদ্ধ"],
            "Mughal/Sultanate Era"  : ["মুঘল", "সম্রাট", "সুলতান"],
            "Modern Politics"       : ["রাষ্ট্রপতি", "প্রধানমন্ত্রী", "সংসদ", "মন্ত্রী", "নির্বাচন"],
            "Independence Movement" : ["স্বাধীনতা", "আন্দোলন", "বিপ্লব"],
        },
    },
    "Geography": {
        "keywords": ["নদী", "পর্বত", "সাগর", "উপজেলা", "জেলা", "বিভাগ", "রাজধানী",
                     "আয়তন", "সীমান্ত", "দ্বীপ", "সুন্দরবন", "হাওর", "উপত্যকা"],
        "topics": {
            "Rivers/Water Bodies"      : ["নদী", "সাগর", "হ্রদ", "উপসাগর", "হাওর"],
            "Administrative Geography" : ["জেলা", "উপজেলা", "বিভাগ", "রাজধানী", "সীমান্ত"],
            "Landforms"                : ["পর্বত", "দ্বীপ", "সুন্দরবন", "উপত্যকা"],
        },
    },
    "Culture_Literature": {
        "keywords": ["কবি", "লেখক", "উপন্যাস", "কবিতা", "সাহিত্য", "নাটক",
                     "রবীন্দ্র", "নজরুল", "চলচ্চিত্র", "বাউল", "লালন", "জারি", "সারি"],
        "topics": {
            "Poetry/Literature" : ["কবি", "কবিতা", "উপন্যাস", "সাহিত্য", "রবীন্দ্র", "নজরুল"],
            "Film/Music"        : ["চলচ্চিত্র", "নাটক"],
            "Folk Tradition"    : ["বাউল", "লালন", "জারি", "সারি"],
        },
    },
    "Language_Grammar": {
        "keywords": ["সমাস", "সন্ধি", "উপসর্গ", "প্রত্যয়", "ব্যাসবাক্য", "বাগধারা",
                     "প্রবাদ", "অর্থ", "ভাবার্থ", "বানান", "বাক্য", "শব্দ"],
        "topics": {
            "Idiom/Proverb Meaning" : ["বাগধারা", "প্রবাদ", "ভাবার্থ"],
            "Word Meaning"          : ["অর্থ", "শব্দের অর্থ", "শাব্দিক"],
            "Grammar - Sandhi"      : ["সন্ধি"],
            "Grammar - Samas"       : ["সমাস", "ব্যাসবাক্য"],
            "Grammar - Affixes"     : ["উপসর্গ", "প্রত্যয়"],
            "Spelling/Usage"        : ["বানান", "শুদ্ধ", "অশুদ্ধ"],
        },
    },
    "Mathematics_Logic": {
        "keywords": ["যোগফল", "বিয়োগ", "গুণ", "ভাগফল", "শতাংশ", "শতকরা", "সমীকরণ", "অসমীকরণ",
                     "অনুপাত", "সমানুপাত", "ত্রিভুজ", "চতুর্ভুজ", "বৃত্ত", "কোণ", "বাহু", "কর্ণ",
                     "ক্ষেত্রফল", "পরিসীমা", "ব্যাসার্ধ", "সংখ্যা", "মৌলিক", "গুণিতক", "উৎপাদক",
                     "লসাগু", "গসাগু", "ভগ্নাংশ", "বর্গমূল", "ঘনফল", "দূরত্ব", "গতিবেগ", "ঘণ্টায়",
                     "কিমি", "কিলোমিটার", "মিটার", "সেকেন্ডে", "কাজ", "চৌবাচ্চা", "নল", "একত্রে",
                     "একসাথে", "লাগবে", "টাকা", "দাম", "মূল্য", "সুদ", "মুনাফা", "লাভ", "ক্ষতি",
                     "আসল", "গড়", "মধ্যক", "প্রচুরক", "বয়স", "জ্বলে", "যথাক্রমে", "এর মান",
                     "মান কত", "মান নির্ণয়", "সমাধান সেট", "সমাধান কোনটি", "অপেক্ষক", "অনুক্রম",
                     "√", "²", "³", "≤", "≥", "∈", "π", "সেট A", "সেট B", "লগ", "x =", "x²", "x³",
                     "এর ল. সা.গু", "এর গ. সা.গু", "×", "%", "<", ">", "ট্রেন", "স্টেশন"],
        "topics": {
            "Work & Time"           : ["কাজ", "চৌবাচ্চা", "নল", "একত্রে", "একসাথে", "লাগবে"],
            "Profit/Interest/Money" : ["সুদ", "মুনাফা", "লাভ", "ক্ষতি", "আসল", "দাম", "মূল্য", "টাকা"],
            "Percentage"            : ["শতকরা", "শতাংশ"],
            "Ratio/Proportion"      : ["অনুপাত", "সমানুপাত"],
            "Average/Statistics"    : ["গড়", "মধ্যক", "প্রচুরক"],
            "Age Problems"          : ["বয়স"],
            "Geometry"              : ["ত্রিভুজ", "চতুর্ভুজ", "বৃত্ত", "কোণ", "বাহু", "কর্ণ", "ক্ষেত্রফল", "পরিসীমা", "ব্যাসার্ধ", "আয়তন"],
            "Sets"                  : ["সেট A", "সেট B", "∈"],
            "Sequences/Series"      : ["অনুক্রম", "ধারাটির", "অনুক্রমটির"],
            "Number Theory/LCM-GCD" : ["মৌলিক", "গুণিতক", "উৎপাদক", "লসাগু", "গসাগু", "জ্বলে", "ভগ্নাংশ", "ল. সা.গু", "গ. সা.গু", "সংখ্যা"],
            "Rate/Time/Speed"       : ["দূরত্ব", "গতিবেগ", "ঘণ্টায়", "কিমি", "কিলোমিটার", "মিটার", "মিনিট", "সেকেন্ডে", "ট্রেন", "স্টেশন"],
            "Algebra/Equations"     : ["সমীকরণ", "অসমীকরণ", "বর্গমূল", "ঘনফল", "এর মান", "মান কত", "মান নির্ণয়", "অপেক্ষক", "√", "²", "³", "≤", "≥", "লগ", "x =", "x²", "x³"],
            "Arithmetic"            : ["যোগফল", "বিয়োগ", "গুণ", "ভাগফল", "×", "%", "<", ">"],
        },
    },
    "Science": {
        "keywords": ["পদার্থ", "তেজস্ক্রীয়", "জীবাণু", "আবিষ্কার",
                     "গ্রহ নক্ষত্র", "বিজ্ঞানী", "রোগ", "শক্তি", "কোষ"],
        "topics": {
            "Chemistry/Elements" : ["পদার্থ", "তেজস্ক্রীয়"],
            "Biology/Medicine"   : ["জীবাণু", "রোগ", "কোষ"],
            "Physics"            : ["শক্তি", "গ্রহ নক্ষত্র"],
            "Scientists/History" : ["আবিষ্কার", "বিজ্ঞানী"],
        },
    },
}
SUMMARIZATION_KEYWORDS = ["সারসংক্ষেপ", "সারমর্ম", "সংক্ষেপে বলুন", "সংক্ষিপ্ত করো"]
REASONING_KEYWORDS = [
    "কতটি", "কতজন", "কত সংখ্যক", "গণনা", "হিসাব",
    "যোগফল", "বিয়োগ", "গুণ", "ভাগ", "শতাংশ", "অনুপাত", "সমীকরণ", "যথাক্রমে",
    "যদি", "তাহলে কত", "প্রমাণ করো", "সমাধান", "নির্ণয় করো",
    "দূরত্ব", "দৈর্ঘ্য", "প্রস্থ", "উচ্চতা", "আয়তন",
    "ঘণ্টা", "ঘণ্টায়", "মিনিট", "কিমি", "কিলোমিটার", "লিটার", "দাম", "টাকা",
    "=", "+", "×", "÷", "%",
]
NEGATION_WORDS = {"নয়", "ছাড়া", "ব্যতীত", "না"}

In [ ]:
def latin_ratio(s):
    letters = re.findall(r"[A-Za-zঀ-৿]", s)
    return 0.0 if not letters else sum(1 for c in letters if c.isascii()) / len(letters)

def detect_language_mix(text):
    r = latin_ratio(text)
    return "English" if r > 0.7 else ("Code-Mixed" if r > 0.15 else "Bangla")

def detect_is_mcq(prompt):
    p = str(prompt)
    bn = set(re.findall(r"(?<![ঀ-৿])[কখগঘঙ][\).]", p))
    en = set(re.findall(r"(?<![A-Za-z])[a-dA-D][\).]", p))
    return len(bn) >= 2 or len(en) >= 2

def _bengali_tokens(text):
    return [t for t in re.split(r"[^ঀ-৿]+", text) if t]

def detect_has_negation(prompt):
    return any(tok in NEGATION_WORDS for tok in _bengali_tokens(prompt))

def detect_has_number_or_date(text):
    return bool(re.search(r"[0-9০-৯]", text))

def detect_reasoning_required(combined):
    return any(kw in combined for kw in REASONING_KEYWORDS)

def detect_domain_subdomain(combined):
    for domain, spec in DOMAIN_TOPICS.items():
        if any(kw in combined for kw in spec["keywords"]):
            for subdomain, sub_kws in spec["topics"].items():
                if any(kw in combined for kw in sub_kws):
                    return domain, subdomain
            return domain, f"{domain} - General"
    return "General_Knowledge", "General_Knowledge - Unclassified"

def detect_task_type(row, is_mcq, reasoning_required, language_mix):
    prompt = str(row.get("prompt_bn", "")); response = str(row.get("response_bn", ""))
    combined = prompt + " " + response
    has_context = str(row.get("context", "[NULL]")).strip() != "[NULL]"
    if any(kw in combined for kw in SUMMARIZATION_KEYWORDS): return "Summarization"
    if is_mcq: return "Multiple-Choice-QA"
    if has_context: return "Contextual-QA"
    if reasoning_required: return "Reasoning-Calculation"
    if language_mix != "Bangla": return "Code-Mixed-QA"
    if any(kw in combined for kw in ["অর্থ", "ভাবার্থ", "শাব্দিক", "বাগধারা", "প্রবাদ", "সমাস", "সন্ধি"]):
        return "Definition-Translation"
    return "Generative-QA"

def complexity_level(is_mcq, has_negation, reasoning_required):
    score = int(is_mcq) + int(has_negation) + int(reasoning_required)
    return "Simple" if score == 0 else ("Medium" if score == 1 else "Complex")

records = []
for _, row in df_test.iterrows():
    prompt = str(row.get("prompt_bn", "")); response = str(row.get("response_bn", ""))
    context = str(row.get("context", "[NULL]")); combined = prompt + " " + response
    hc  = context.strip() != "[NULL]"
    lm  = detect_language_mix(combined)
    mcq = detect_is_mcq(prompt)
    neg = detect_has_negation(prompt)
    num = detect_has_number_or_date(combined)
    rr  = detect_reasoning_required(combined)
    dom, sub = detect_domain_subdomain(combined)
    records.append({
        "id": row["id"], "has_context": hc, "task_type": detect_task_type(row, mcq, rr, lm),
        "domain": dom, "subdomain": sub, "complexity_level": complexity_level(mcq, neg, rr),
        "language_mix": lm, "is_mcq": mcq, "has_negation": neg, "has_number_or_date": num,
    })
feat = pd.DataFrame(records)
df_test = df_test.merge(feat, on="id")
print("Categorized. task_type counts:")
print(df_test["task_type"].value_counts())

## 4. Factual-KB Hint (fuzzy question -> reference answer)

Char n-gram TF-IDF matches each question against a Bengali Q->A reference. On a close match
(`sim >= GOLD_SIM_TH`), the reference answer is attached and later passed to the prompt as
evidence for the LLM to judge against. Offline, CPU-only, no GPU.

In [ ]:
_BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

def _norm(s):
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"[?？।\.\!,;:\"'`()\[\]{}]+", " ", s)
    return re.sub(r"\s+", " ", s).strip().lower()

def _norm_ans(s):
    return re.sub(r"\s+", " ", _norm(s).translate(_BN_DIGITS)).strip()

def _nums(s):
    return set(re.findall(r"\d+", _norm_ans(s)))

def answer_matches_response(gold, resp):
    """Deterministic verdict for a covered question: 1 (faithful) if the Response is
    consistent with the verified gold answer, else 0. Handles Bengali digits, numeric
    equality, containment, and high token overlap."""
    g, r = _norm_ans(gold), _norm_ans(resp)
    if not g or not r:
        return None
    if g == r or g in r or r in g:
        return 1
    gn = _nums(g)
    if gn:
        return 1 if gn & _nums(r) else 0
    gt, rt = set(g.split()), set(r.split())
    if gt and len(gt & rt) / len(gt) >= 0.6:
        return 1
    return 0

if USE_GOLD_HINT and os.path.exists(GOLD_KB_PATH):
    kb = pd.read_csv(GOLD_KB_PATH)
    kb_norm = [_norm(q) for q in kb["question"]]
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)
    G = vec.fit_transform(kb_norm)
    T = vec.transform([_norm(q) for q in df_test["prompt_bn"]])
    best_sim = np.zeros(len(df_test)); best_idx = np.zeros(len(df_test), dtype=int)
    for i in range(0, T.shape[0], 512):
        s = (T[i:i+512] @ G.T).toarray()
        best_sim[i:i+512] = s.max(axis=1); best_idx[i:i+512] = s.argmax(axis=1)
    df_test["gold_hint"] = [str(kb.iloc[best_idx[j]]["correct_answer"]) if best_sim[j] >= GOLD_SIM_TH else ""
                            for j in range(len(df_test))]
    print("Factual Knowledge base attached.")
else:
    df_test["gold_hint"] = ""
    print("Factual-KB hint disabled or KB not found -- proceeding without it.")

def _ni(s):
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"[\"'“”‘’`।?\.\!,;:()\[\]{}]+", " ", s)
    return re.sub(r"\s+", " ", s).strip().lower()

def _extract_phrase(q):
    q = str(q)
    m = re.search(r"[\"“‘']([^\"”’']{1,40})[\"”’']", q)
    if m: return m.group(1).strip()
    m = re.search(r"^(.{1,40}?)\s*(?:এর|-এর)\s*(?:ভাবার্থ|শাব্দিক|অর্থ|মানে)", q)
    if m: return m.group(1).strip()
    return ""

if USE_IDIOM_HINT and os.path.exists(IDIOM_KB_PATH):
    ik = pd.read_csv(IDIOM_KB_PATH)
    _idiom_lookup = {}
    for _idm, _mean in zip(ik["idiom"], ik["figurative_meaning_bn"]):
        _idiom_lookup.setdefault(_ni(_idm), str(_mean))
    _literal_lookup = {}
    for _idm, _lit in zip(ik["idiom"], ik["literal_meaning"]):
        _literal_lookup.setdefault(_ni(_idm), str(_lit))
    def _idiom_hint(q):
        if not re.search(r"অর্থ|ভাবার্থ|মানে|কী\s*বোঝায়|শাব্দিক", str(q)):
            return ""
        return _idiom_lookup.get(_ni(_extract_phrase(q)), "")
    def _literal_hint(q):
        if not re.search(r"অর্থ|ভাবার্থ|মানে|কী\s*বোঝায়|শাব্দিক", str(q)):
            return ""
        return _literal_lookup.get(_ni(_extract_phrase(q)), "")
    df_test["idiom_hint"] = [_idiom_hint(q) for q in df_test["prompt_bn"]]
    df_test["literal_hint"] = [_literal_hint(q) for q in df_test["prompt_bn"]]
    print("Idiom knowledge base attached.")
else:
    df_test["idiom_hint"] = ""
    df_test["literal_hint"] = ""
    print("Idiom hint disabled or KB not found -- proceeding without idiom hints.")

if USE_GRAMMAR_HINT and os.path.exists(GRAMMAR_KB_PATH):
    gk = pd.read_csv(GRAMMAR_KB_PATH)
    gk_norm = [_norm(q) for q in gk["question"]]
    _gvec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)
    _G2 = _gvec.fit_transform(gk_norm)
    _T2 = _gvec.transform([_norm(q) for q in df_test["prompt_bn"]])
    _gsim = np.zeros(len(df_test)); _gidx = np.zeros(len(df_test), dtype=int)
    for i in range(0, _T2.shape[0], 512):
        s = (_T2[i:i+512] @ _G2.T).toarray()
        _gsim[i:i+512] = s.max(axis=1); _gidx[i:i+512] = s.argmax(axis=1)
        _GRAMMAR_Q = re.compile(r"সন্ধি|সমাস|উপসর্গ|প্রত্যয়|বিচ্ছেদ|ব্যাসবাক্য|কারক|বিভক্তি|ধাতু|লিঙ্গ|বচন")
    _GQ = "['\"\u201c\u201d\u2018\u2019]"
    def _gquoted(q):
        q = unicodedata.normalize("NFC", str(q))
        return set(re.findall(r"[\u0980-\u09ff]{2,20}(?=" + _GQ + ")", q)
                   + re.findall("(?<=" + _GQ + ")[\u0980-\u09ff]{2,20}", q))
    def _gkeep(test_q, kb_q):
        tw = _gquoted(test_q)
        if not tw:
            return True
        kn = _norm(kb_q)
        return all(w in kn for w in tw)
    _ghints = []
    for j in range(len(df_test)):
        _q = df_test["prompt_bn"].iloc[j]
        if (_gsim[j] >= GRAMMAR_SIM_TH
                and _GRAMMAR_Q.search(unicodedata.normalize("NFC", str(_q)))
                and _gkeep(_q, gk.iloc[_gidx[j]]["question"])):
            _ghints.append(str(gk.iloc[_gidx[j]]["correct_answer"]))
        else:
            _ghints.append("")
    df_test["grammar_hint"] = _ghints
    print("Grammar knowledge base attached.")
else:
    df_test["grammar_hint"] = ""
    print("Grammar hint disabled or grammar_kb.csv not found -- proceeding without it "
          "(sandhi/samas/affix questions still get the meaning-tier's dedicated grammar prompt, "
          "just without a KB-verified reference answer).")

import glob as _glob_sp
SPELL_DICT = None
try:
    if not USE_SPELL_DICT:
        raise RuntimeError('USE_SPELL_DICT is False')
    _sp_files = []
    for _pat in SPELL_DICT_GLOBS:
        _sp_files += _glob_sp.glob(_pat, recursive=True)
    _sd = set()
    for _p in _sp_files:
        for _w in open(_p, encoding='utf-8-sig'):
            _w = unicodedata.normalize('NFC', _w.strip())
            if _w:
                _sd.add(_w)
    if _sd:
        SPELL_DICT = _sd
        print('Spelling dictionary attached.')
    else:
        print('Spelling dictionary not attached -- spelling override disabled (no effect).')
except Exception as _e:
    print(f'Spelling dictionary load skipped ({_e}) -- override disabled.')


## 4b. RAG retrieval + rerank (advisory evidence for NO-CONTEXT questions)

Runs **before** the LLM loads (so the embedder/reranker are freed before Gemma-3-12B takes VRAM).
For every no-context question:
1. **Dense retrieve** top-`RAG_N_CAND` passages from the prebuilt corpus (BGE-M3 embeddings,
   torch matmul — no faiss dependency).
2. **Cross-encoder rerank** those candidates with `bge-reranker-v2-m3` and keep only the top
   `RAG_TOPK` whose score clears `RAG_RERANK_MIN`. The reranker is what keeps noise out — a
   passage that isn't truly relevant scores below the floor and is dropped.
3. The survivors become **advisory evidence** injected into the prompt; the LLM still decides.

Fully **graceful**: if the RAG dataset / model weights aren't attached, this cell prints a
notice and the notebook runs exactly as before (no RAG).

In [ ]:
df_test["rag_evidence"] = ""
_rag_ok = USE_RAG and all(os.path.exists(p) for p in [RAG_EMB_NPY, RAG_CHUNKS, RAG_EMBED_PATH, RAG_RERANK_PATH])
if not _rag_ok:
    print("RAG not active (dataset/model not attached or USE_RAG=False) -- running without retrieval.")
else:
    try:
        from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
        import torch.nn.functional as F

        def _embed(texts, tok, mdl, bs=64, maxlen=384):
            outs = []
            for i in range(0, len(texts), bs):
                enc = tok(texts[i:i+bs], padding=True, truncation=True, max_length=maxlen,
                          return_tensors="pt").to(DEVICE)
                with torch.no_grad():
                    h = mdl(**enc).last_hidden_state[:, 0]
                outs.append(F.normalize(h, dim=1).half().cpu())
            return torch.cat(outs)

        corpus_emb = torch.from_numpy(np.load(RAG_EMB_NPY)).half().to(DEVICE)
        corpus = pd.read_parquet(RAG_CHUNKS); corpus_txt = corpus["text"].astype(str).tolist()
        print(f"RAG corpus: {corpus_emb.shape[0]} chunks")

        mask = ~df_test["has_context"].astype(bool)
        qpos = df_test.index[mask].tolist()
        qs   = df_test.loc[mask, "prompt_bn"].astype(str).tolist()
        etok = AutoTokenizer.from_pretrained(RAG_EMBED_PATH)
        emdl = AutoModel.from_pretrained(RAG_EMBED_PATH).to(DEVICE).half().eval()
        qemb = _embed(qs, etok, emdl).to(DEVICE)
        del emdl; gc.collect(); torch.cuda.empty_cache()

        cand = []
        for i in range(0, qemb.shape[0], 128):
            sims = qemb[i:i+128] @ corpus_emb.T
            cand.append(sims.topk(RAG_N_CAND, dim=1).indices.cpu())
        cand = torch.cat(cand)
        del corpus_emb, qemb; gc.collect(); torch.cuda.empty_cache()

        rtok = AutoTokenizer.from_pretrained(RAG_RERANK_PATH)
        rmdl = AutoModelForSequenceClassification.from_pretrained(RAG_RERANK_PATH).to(DEVICE).half().eval()
        n_hit = 0
        for r, gi in enumerate(tqdm(qpos, desc="RAG rerank")):
            idxs = cand[r].tolist()
            ctxts = [corpus_txt[j] for j in idxs]
            pairs = [[qs[r], c] for c in ctxts]
            with torch.no_grad():
                enc = rtok(pairs, padding=True, truncation=True, max_length=512,
                           return_tensors="pt").to(DEVICE)
                sc = rmdl(**enc).logits.view(-1).float().cpu()
            order = sc.argsort(descending=True)
            keep = [ctxts[o][:RAG_MAX_CHARS] for o in order[:RAG_TOPK] if sc[o] >= RAG_RERANK_MIN]
            if keep:
                df_test.at[gi, "rag_evidence"] = "\n---\n".join(keep); n_hit += 1
        try: del enc, sc
        except NameError: pass
        del rmdl, rtok; gc.collect(); torch.cuda.empty_cache()
        print(f"RAG: retrieved+reranked evidence for {n_hit}/{len(qpos)} no-context questions.")
        print(f"  VRAM after RAG cleanup: allocated={torch.cuda.memory_allocated()/1e9:.2f}GB "
              f"reserved={torch.cuda.memory_reserved()/1e9:.2f}GB")
    except Exception as e:
        print("RAG step failed, continuing WITHOUT retrieval:", repr(e))
        df_test["rag_evidence"] = ""

## 5. Lean Prompt Builder (speed-first) + verdict parser

Two prompt styles:
- **fast** (non-math): a short, direct verdict prompt — no few-shot, no long checklist, no
  `<think>`. For **context** questions it strictly asks whether the Response is *supported by
  and consistent with the Context*. For no-context it asks a direct factual check. Answers in
  one line, so decoding is short.
- **reasoning** (only Reasoning-Calculation): a trimmed thinking prompt for step-by-step math.

In [ ]:
_AT_HINT = {
    "date_year"  : "The answer must be the exact correct year/date.",
    "person_name": "The answer must be the exact correct person/name.",
    "place"      : "The answer must be the exact correct place/location.",
    "quantity"   : "The answer must be the exact correct number/quantity.",
    "meaning"    : "Judge the meaning; for idioms (বাগধারা) use the FIGURATIVE meaning, not literal.",
    "general"    : "",
}

def infer_answer_type(prompt_bn):
    p = str(prompt_bn)
    if re.search(r"কবে|কোন\s*সালে|কত\s*সালে|কোন\s*সময়|কত\s*সাল|কোন\s*তারিখ", p): return "date_year"
    if re.search(r"কোথায়|কোন\s*স্থানে|কোন\s*জেলায়|কোন\s*দেশে|কোন\s*শহরে|অবস্থিত", p): return "place"
    if re.search(r"কতটি|কতজন|কত\s*সংখ্যক|কয়টি|সংখ্যা|শতাংশ|অনুপাত|কত\b", p): return "quantity"
    if re.search(r"অর্থ|ভাবার্থ|মানে|কাকে\s*বলে|কী\s*বোঝায়|শাব্দিক", p): return "meaning"
    if re.search(r"\bকে\b|কারা|কে\s*ছিলেন|কে\s*হন|রচয়িতা|লেখক|আবিষ্কার|প্রতিষ্ঠাতা|জনক", p): return "person_name"
    return "general"

_MEANING_RE = re.compile(r"ভাবার্থ|শাব্দিক|বাগধারা|প্রবাদ|এর\s*অর্থ|শব্দের\s*অর্থ|কাকে\s*বলে|কী\s*বোঝায়|মানে\s*কী|"
                         r"সন্ধি|সমাস|ব্যাসবাক্য|উপসর্গ|প্রত্যয়|বিচ্ছেদ")
_CALC_RE    = re.compile(r"নির্ণয়|কত\s*হবে|যোগফল|গুণফল|ভাগফল|শতকরা|শতাংশ|%|সুদ|মুনাফা|লসাগু|গসাগু|"
                         r"সমীকরণ|অনুপাত|গড়|ক্ষেত্রফল|পরিসীমা|বর্গমূল|ঘনফল|সম্ভাবনা")

def route_tier(prompt, has_ctx):
    p = str(prompt)
    if _MEANING_RE.search(p):
        return "meaning"
    digits = len(re.findall(r"[0-9০-৯]", p))
    if re.search(r"যথাক্রমে|একত্রে", p) and digits >= 2:
        return "reasoning"
    if re.search(r"\bx\b\s*=|=\s*[\dxX]|x\s*এর\s*মান|x[²³2-9]|সমীকরণ|সেট\s*[A-Zক-হ]?\s*=", p):
        return "reasoning"
    if re.search(r"বয়স", p) and digits >= 2:
        return "reasoning"
    if re.search(r"কাজ", p) and re.search(r"দিন|ঘণ্টা|ঘণ্টায়|জন", p) and digits >= 2:
        return "reasoning"
    if re.search(r"ত্রিভুজ|বৃত্ত|ব্যাসার্ধ|কর্ণ|সমবাহু|সমকোণ|বাহু|পঞ্চভুজ|ষড়ভুজ|বহুভুজ|চতুর্ভুজ", p) and (digits >= 1 or "মান" in p or "সমষ্টি" in p):
        return "reasoning"
    if re.search(r"অঙ্ক", p) and digits >= 1:
        return "reasoning"
    if re.search(r"ঘড়ি", p) and re.search(r"সেকেন্ড|মিনিট", p):
        return "reasoning"
    if re.search(r"ট্রেন", p) and re.search(r"স্টেশন", p):
        return "reasoning"
    if re.search(r"রূপান্তর", p):
        return "reasoning"
    if re.search(r"গুণ", p) and re.search(r"বাড়বে|কমবে|কত\s*গুণ", p):
        return "reasoning"
    if re.search(r"\u221a|\u00d7|\u03c0", p):
        return "reasoning"
    if re.search(r"(?<![0-9\u09e6-\u09ef])[<>\u2264\u2265](?![0-9\u09e6-\u09ef]*%)", p):
        return "reasoning"
    if re.search(r"\b[0-9\u09e6-\u09ef]+\s*/\s*[0-9\u09e6-\u09ef]+\b", p):
        return "reasoning"
    if _CALC_RE.search(p) and not has_ctx:
        return "reasoning"
    return "fast"

def _meta_block(row, kind):
    # task_type reflects the ACTUAL processing tier so the metadata never contradicts the
    # prompt style (e.g. a work-rate word-problem handled in the reasoning tier is labelled
    # Reasoning-Calculation even when the coarse rule-based tag said Generative-QA).
    task_type = {"reasoning": "Reasoning-Calculation", "meaning": "Definition-Meaning",
                 "goldref": "Factual-QA"}.get(kind, str(row.get('task_type', '')))
    parts = [
        f"domain={row.get('domain','')}", f"subdomain={row.get('subdomain','')}",
        f"task_type={task_type}", f"complexity={row.get('complexity_level','')}",
        f"language_mix={row.get('language_mix','')}", f"is_mcq={bool(row.get('is_mcq'))}",
        f"has_negation={bool(row.get('has_negation'))}",
        f"has_number_or_date={bool(row.get('has_number_or_date'))}",
    ]
    return ("Metadata: " + " | ".join(parts) + '\n' +
            "(Consider these signals -- domain, task type, complexity, negation, and whether "
            "numeric/date content is involved -- before analyzing the Question and Response below, "
            "then decide.)" + '\n\n')

def build_messages(row, kind):
    q = str(row.get("prompt_bn", "")); a = str(row.get("response_bn", ""))
    has_ctx = bool(row.get("has_context"))
    ctx = str(row.get("context", "[NULL]"))
    gold = str(row.get("gold_hint", "")).strip()
    gold_line = (f"A verified correct answer to this question (from a trusted "
                 f"knowledge base) is: \"{gold}\". Use it as strong evidence and judge the "
                 f"Response STRICTLY against it: output 1 (FAITHFUL) ONLY if the Response conveys "
                 f"this same answer (exact match OR an unambiguous paraphrase/synonym); output 0 "
                 f"(HALLUCINATION) if it gives a different, conflicting, or contradictory answer. "
                 f"Do not accept a Response that disagrees with this verified answer, even if it "
                 f"sounds confident.\n") if gold else ""
    rag = str(row.get("rag_evidence", "")).strip()
    rag_block = (f"Reference material (retrieved; may be incomplete or unrelated -- use it ONLY "
                 f"if it clearly bears on the Question, otherwise rely on your own knowledge):\n"
                 f"{rag}\n\n") if rag else ""
    meta_block = _meta_block(row, kind)

    if kind == "reasoning":
        ctx_block = f"Context: {ctx}\n\n" if has_ctx else ""
        content = (
            "You are a math expert. Verify this Bengali word-problem answer with precision, "
            "but write CONCISELY.\n\n"
            f"{ctx_block}{rag_block}{meta_block}Question: {q}\nResponse: {a}\n\n{gold_line}"
            "In <think></think>: pick the right method "
            "(work-rate: 1/A+1/B+...=1/together; average: total/count; ratio: solve "
            "proportionally; SI=P*R*T/100; ages/LCM/GCD: set up the equation), compute, then "
            "VERIFY before concluding. Only key equations/numbers -- no restating the problem.\n"
            "After </think>, output exactly one line:\n"
            "[LABEL]: 1  (Response matches your verified answer) or [LABEL]: 0  (Response is wrong)"
        )
        return [{"role": "user", "content": content}]

    if kind == "meaning":
        is_grammar    = bool(re.search(r"সন্ধি|সমাস|ব্যাসবাক্য|উপসর্গ|প্রত্যয়|বিচ্ছেদ", q))
        is_figurative = bool(re.search(r"বাগধারা|প্রবাদ|ভাবার্থ", q))

        meaning_calib = ("Be STRICT: unless the Response clearly conveys the SAME meaning, "
                         "choose 0. A vague, incomplete, or slightly-different meaning "
                         "is a hallucination (0)." if CALIBRATION == "strict" else "")

        if is_grammar:
            grammar_hint = str(row.get("grammar_hint", "")).strip()
            hint_line = (f"A trusted grammar reference gives the correct answer as: \"{grammar_hint}\". "
                        f"Treat this as ground truth.\n\n") if grammar_hint else ""
            content = (
                "You are an expert in Bengali (বাংলা) grammar: সন্ধি (sandhi/euphonic combination), "
                "সমাস (compound-word classification), উপসর্গ/প্রত্যয় (prefixes/suffixes).\n\n"
                "Worked example (format only -- NOT the question to judge):\n"
                "Question: 'রাজপুত্র' শব্দের সন্ধি বিচ্ছেদ কী?\n"
                "Response: রাজ + পুত্র\n"
                "Check: 'রাজপুত্র' = রাজ + পুত্র (স্বরসন্ধি)। Response সঠিক।\n"
                "[LABEL]: 1\n\n"
                "Now judge the ACTUAL question:\n"
                f"{rag_block}{meta_block}Question: {q}\nResponse: {a}\n\n{hint_line}{gold_line}"
                "Work out the correct grammatical breakdown/classification step by step, then compare "
                "it EXACTLY (the specific term or breakdown, not just 'same general topic') against "
                "the Response. A response about the right topic but the WRONG specific term or "
                "breakdown is still a hallucination (0).\n"
                f"{meaning_calib}\n"
                "Answer in 2-3 short sentences ONLY -- no markdown headers, no bold (**text**), "
                "no restating the question, no preamble like 'Let's analyze'. State the correct "
                "term/breakdown and compare it directly against the Response.\n"
                "For উপসর্গ class, these name the SAME class -- treat as equal, not a mismatch: "
                "'বাংলা' = 'খাঁটি বাংলা' = 'খাঁটি বাংলা উপসর্গ'; 'তৎসম' = 'সংস্কৃত'; "
                "'বিদেশি' = 'বিদেশী'. Judge on the class itself, not on the short/long wording.\n"
                "For সমাস, the trailing word 'সমাস' and hyphens are OPTIONAL -- treat these as "
                "the same: 'দ্বন্দ্ব' = 'দ্বন্দ্ব সমাস'; 'সপ্তমী তৎপুরুষ' = 'সপ্তমী-তৎপুরুষ' "
                "= 'সপ্তমী তৎপুরুষ সমাস'. Match on the সমাস type, not the exact wording.\n"
                "Output your check, then the final line EXACTLY:\n"
                "[LABEL]: 1  (exact match) or [LABEL]: 0  (wrong term/breakdown)"
            )
            return [{"role": "user", "content": content}]

        if is_figurative:
            idiom = str(row.get("idiom_hint", "")).strip()
            idiom_line = (f"A trusted idiom dictionary gives the correct meaning as: \"{idiom}\". "
                          f"Treat this as the ground-truth meaning.\n\n") if idiom else ""
            content = (
                "You are an expert in Bengali idioms (বাগধারা) and proverbs.\n\n"
                "Two worked examples (format only -- NOT the question to judge):\n"
                "Question: \"গোঁফ খেজুরে\" এর ভাবার্থ কী?\n"
                "Response: অত্যন্ত অলস ব্যক্তি\n"
                "Meaning: \"গোঁফ খেজুরে\" means an extremely lazy person; the Response conveys the same idea.\n"
                "[LABEL]: 1\n\n"
                "Question: \"গোঁফ খেজুরে\" এর ভাবার্থ কী?\n"
                "Response: খুব সাহসী মানুষ\n"
                "Meaning: it means a lazy person, NOT a brave one -- the Response states a different meaning.\n"
                "[LABEL]: 0\n\n"
                "Now judge the ACTUAL question:\n"
                f"{rag_block}{meta_block}Question: {q}\nResponse: {a}\n\n{idiom_line}{gold_line}"
                "Step 1: state the correct FIGURATIVE meaning of the phrase in ONE short sentence "
                "(not a literal word-by-word translation).\n"
                "Step 2: does the Response convey that SAME meaning? A paraphrase or synonym counts as "
                "CORRECT (1). But if the Response gives a DIFFERENT meaning, or only the literal sense, "
                "it is a hallucination (0) -- do NOT accept a wrong meaning just because it is "
                "confidently written.\n"
                f"{meaning_calib}\n"
                "DECISION RULE: compare the Response to the meaning YOU stated in Step 1. If they express the same core idea -- a synonym, a paraphrase, or different wording all count as a match -- output 1. Output 0 ONLY if the Response gives a genuinely different, opposite, or wrong meaning; do NOT output 0 merely for different wording or slightly less detail.\n"
                "Output your Step-1 meaning sentence, then the final line EXACTLY:\n"
                "[LABEL]: 1  (matches) or [LABEL]: 0  (different/wrong meaning)"
            )
            return [{"role": "user", "content": content}]

        literal_hint = str(row.get("literal_hint", "")).strip()
        fig_hint = str(row.get("idiom_hint", "")).strip()
        sense_ref = ""
        if literal_hint:
            sense_ref += f"A dictionary reference gives the literal meaning as: \"{literal_hint}\".\n"
        if fig_hint:
            sense_ref += (f"This word also has an established figurative/symbolic meaning: "
                         f"\"{fig_hint}\".\n")
        if sense_ref:
            if fig_hint:
                sense_ref += ("Treat both as trusted references. For an idiom or set expression, BOTH "
                             "the literal and the established figurative meaning are acceptable answers "
                             "-- mark the Response CORRECT (1) if it conveys EITHER of them (exact match "
                             "or a reasonable paraphrase/synonym), even when the question says শাব্দিক. "
                             "Mark it 0 ONLY if the Response gives a meaning that matches NEITHER sense "
                             "(a genuinely different or wrong meaning).\n\n")
            else:
                sense_ref += ("Treat this as a trusted reference -- the Response is correct if it "
                             "matches this meaning (exact match or paraphrase).\n\n")
        content = (
            "You are an expert Bengali lexicographer judging word meanings. A question phrased as "
            "\"শাব্দিক অর্থ\" (literal meaning) usually wants the literal/dictionary sense, but some "
            "words are conventionally used in an established figurative or symbolic sense instead -- "
            "judge the Response against whichever sense is actually correct for THIS word.\n\n"
            "Worked example (format only -- NOT the question to judge):\n"
            "Question: \"ফ্ল্যাট\" এর শাব্দিক অর্থ কী?\n"
            "Response: চ্যাপ্টা বা সমতল\n"
            "Check: এটি সাধারণ শব্দের আক্ষরিক/অভিধানগত অর্থ জিজ্ঞাসা। 'চ্যাপ্টা বা সমতল' সঠিক আক্ষরিক অর্থ।\n"
            "[LABEL]: 1\n\n"
            "Now judge the ACTUAL question:\n"
            f"{rag_block}{meta_block}Question: {q}\nResponse: {a}\n\n{sense_ref}{gold_line}"
            "Step 1: state the correct meaning in ONE short sentence -- literal, or the established "
            "figurative/symbolic sense if that is what this word conventionally means.\n"
            "Step 2: does the Response convey that SAME meaning (literal OR figurative, whichever is "
            "correct for this word)? A synonym or paraphrase counts as correct (1).\n"
            f"{meaning_calib}\n"
            "DECISION RULE: compare the Response to the meaning YOU stated in Step 1. If they express the same core idea -- a synonym, a paraphrase, or different wording all count as a match -- output 1. Output 0 ONLY if the Response gives a genuinely different, opposite, or wrong meaning; do NOT output 0 merely for different wording or slightly less detail.\n"
            "Output your Step-1 meaning sentence, then the final line EXACTLY:\n"
            "[LABEL]: 1  (matches) or [LABEL]: 0  (different/wrong meaning)"
        )
        return [{"role": "user", "content": content}]

    at = _AT_HINT.get(infer_answer_type(q), "")
    hint = (" " + at) if at else ""
    if has_ctx:
        ctx_block = f"Context: {ctx}\n\n"
        instr = ("Is the Response SUPPORTED BY the Context above (different wording is fine)? "
                 "Flag it (0) only if it contradicts the Context or adds facts not found in it.")
    else:
        ctx_block = ""
        instr = ("Using your knowledge, is the Response factually CORRECT? A wrong fact, date, "
                 "name, place or number is a hallucination (0).")
    if CALIBRATION == "strict":
        calib = (" Do not assume correctness -- verify first. Choose 0 if wrong/fabricated/unsupported; "
                 "choose 1 only when confirmed correct.")
    elif CALIBRATION == "lean_faithful":
        calib = (" Choose 0 only if genuinely confident it is wrong/unsupported; correct, plausible, "
                 "or paraphrased responses get 1 -- do not flag for brevity or different wording.")
    else:
        calib = ""
    content = (
        "You are an expert Bengali fact-checker.\n\n"
        f"{ctx_block}{rag_block}Question: {q}\nResponse: {a}\n\n{gold_line}{instr}{hint}{calib}\n"
        "Work out the correct answer yourself in 1-2 short sentences, then on the FINAL line "
        "output EXACTLY:\n"
        "[LABEL]: 1 (faithful) or [LABEL]: 0 (hallucinated)."
    )
    return [{"role": "user", "content": content}]

def parse_verdict(text):
    t = unicodedata.normalize("NFC", text)
    m = re.findall(r"\[(?:LABEL|FACT|FAITHFUL|VERDICT|LABELED)\]\s*[:：]?\s*([01])", t)
    if m: return int(m[-1])
    lines = [l.strip() for l in t.strip().splitlines() if l.strip()]
    if lines:
        mm = re.search(r"\b([01])\b", lines[-1])
        if mm: return int(mm.group(1))
    tl = t.lower()
    EN = ["hallucinat", "not faithful", "fabricat", "incorrect", "unsupported", "wrong",
          "does not match", "does not accurately", "incorrectly"]
    BN = [unicodedata.normalize("NFC", w) for w in ["সত্য নয়", "ভুল", "সঠিক নয়", "ঠিক নয়", "অসত্য"]]
    if any(w in tl for w in EN) or any(w in t for w in BN): return 0
    return 1

## 6. Model Load / Inference (Unsloth, Gemma-3-12B, DUAL-GPU data-parallel)

Instead of one model instance processing all rows sequentially on GPU 0, this section:
1. Assigns every row to a tier (fast/meaning/reasoning) -- same `route_tier`/`_pick_tier`
   logic as the single-GPU notebook.
2. Splits the tier-tagged rows into two **balanced** shards (by an estimated per-row workload,
   not just row count, so neither GPU sits idle waiting for the other).
3. Writes a standalone `dual_gpu_worker.py` to disk (self-contained copy of `build_messages`,
   `route_tier`, `parse_verdict`, `load_model`, `_gen_or_split`, `run_inference`, etc. -- the
   exact same tested code as the single-GPU notebook, just runnable as an independent script).
4. Launches **two OS subprocesses** via `subprocess.Popen`, one with `CUDA_VISIBLE_DEVICES=0`,
   one with `CUDA_VISIBLE_DEVICES=1` -- each sees only ITS OWN GPU as "device 0" internally, so
   the worker code needs zero GPU-selection logic. No shared CUDA context between them -> none of
   vLLM's tensor-parallel dtype/subprocess-handshake problems apply here.
5. Waits for both to finish, then merges their per-tier-per-GPU checkpoint files into the same
   `results` dict the rest of the notebook (assembly loop, `reasoning.txt` export, submission)
   already expects -- **zero changes needed downstream**.

**If only one GPU is visible** (e.g. Kaggle accelerator set to a single T4/P100), this section
automatically falls back to the original single-process path -- safe to run either way.


In [ ]:
# Writes the standalone dual-GPU worker script to disk. Re-run this cell if you change any
# prompt/config logic upstream (cell 3's config or cell 14's build_messages) -- it always
# regenerates the worker from the CURRENT in-notebook source, so the two never drift apart.
WORKER_PATH = "dual_gpu_worker.py"
_worker_code = '"""Standalone dual-GPU worker (auto-generated by the notebook -- do not hand-edit; re-run the\nnotebook\'s writer cell instead). Loads ONE Gemma-3-12B copy on whichever GPU\nCUDA_VISIBLE_DEVICES exposes (set by the LAUNCHING process, not this script), processes its\nassigned shard of rows tier-by-tier, and writes results to per-tier-per-shard checkpoint files\n(cache_<tier>_<tag>.json) so a crash only loses the in-flight batch, exactly like the single-GPU\nnotebook\'s resume design.\n"""\nimport os, sys, argparse, re, gc, json, time, unicodedata\n\nap = argparse.ArgumentParser()\nap.add_argument("--shard", required=True)\nap.add_argument("--config", required=True)\nap.add_argument("--tag", required=True)\nargs = ap.parse_args()\n\ncfg = json.load(open(args.config, encoding="utf-8"))\nMODEL_NAME         = cfg["MODEL_NAME"]\nMAX_SEQ_LENGTH     = cfg["MAX_SEQ_LENGTH"]\nMAX_INPUT_LEN      = cfg["MAX_INPUT_LEN"]\nSEQ_TOKEN_BUDGET   = cfg["SEQ_TOKEN_BUDGET"]\nBATCH_SIZE         = cfg["BATCH_SIZE"]\nBATCH_SIZE_MEANING = cfg["BATCH_SIZE_MEANING"]\nBATCH_SIZE_MATH    = cfg["BATCH_SIZE_MATH"]\nCALIBRATION        = cfg["CALIBRATION"]\nSEED               = cfg["SEED"]\nGEN_CONFIG         = cfg["GEN_CONFIG"]\nREASONING_KINDS    = set(cfg["REASONING_KINDS"])\n\nos.environ.setdefault("HF_HUB_DISABLE_XET", "1")\nimport unsloth\nfrom unsloth import FastLanguageModel\nimport torch\nimport pandas as pd\nfrom tqdm.auto import tqdm\nfrom transformers import StoppingCriteria, StoppingCriteriaList\n\nDEVICE = "cuda" if torch.cuda.is_available() else "cpu"\ntorch.manual_seed(SEED)\nprint(f"[{args.tag}] GPUs visible to this process: {torch.cuda.device_count()}", flush=True)\nif torch.cuda.is_available():\n    print(f"[{args.tag}] using: {torch.cuda.get_device_name(0)}", flush=True)\n\n_AT_HINT = {\n    "date_year"  : "The answer must be the exact correct year/date.",\n    "person_name": "The answer must be the exact correct person/name.",\n    "place"      : "The answer must be the exact correct place/location.",\n    "quantity"   : "The answer must be the exact correct number/quantity.",\n    "meaning"    : "Judge the meaning; for idioms (বাগধারা) use the FIGURATIVE meaning, not literal.",\n    "general"    : "",\n}\n\ndef infer_answer_type(prompt_bn):\n    p = str(prompt_bn)\n    if re.search(r"কবে|কোন\\s*সালে|কত\\s*সালে|কোন\\s*সময়|কত\\s*সাল|কোন\\s*তারিখ", p): return "date_year"\n    if re.search(r"কোথায়|কোন\\s*স্থানে|কোন\\s*জেলায়|কোন\\s*দেশে|কোন\\s*শহরে|অবস্থিত", p): return "place"\n    if re.search(r"কতটি|কতজন|কত\\s*সংখ্যক|কয়টি|সংখ্যা|শতাংশ|অনুপাত|কত\\b", p): return "quantity"\n    if re.search(r"অর্থ|ভাবার্থ|মানে|কাকে\\s*বলে|কী\\s*বোঝায়|শাব্দিক", p): return "meaning"\n    if re.search(r"\\bকে\\b|কারা|কে\\s*ছিলেন|কে\\s*হন|রচয়িতা|লেখক|আবিষ্কার|প্রতিষ্ঠাতা|জনক", p): return "person_name"\n    return "general"\n\n_MEANING_RE = re.compile(r"ভাবার্থ|শাব্দিক|বাগধারা|প্রবাদ|এর\\s*অর্থ|শব্দের\\s*অর্থ|কাকে\\s*বলে|কী\\s*বোঝায়|মানে\\s*কী|"\n                         r"সন্ধি|সমাস|ব্যাসবাক্য|উপসর্গ|প্রত্যয়|বিচ্ছেদ")\n_CALC_RE    = re.compile(r"নির্ণয়|কত\\s*হবে|যোগফল|গুণফল|ভাগফল|শতকরা|শতাংশ|%|সুদ|মুনাফা|লসাগু|গসাগু|"\n                         r"সমীকরণ|অনুপাত|গড়|ক্ষেত্রফল|পরিসীমা|বর্গমূল|ঘনফল|সম্ভাবনা")\n\ndef route_tier(prompt, has_ctx):\n    p = str(prompt)\n    if _MEANING_RE.search(p):\n        return "meaning"\n    digits = len(re.findall(r"[0-9০-৯]", p))\n    if re.search(r"যথাক্রমে|একত্রে", p) and digits >= 2:\n        return "reasoning"\n    if re.search(r"\\bx\\b\\s*=|=\\s*[\\dxX]|x\\s*এর\\s*মান|x[²³2-9]|সমীকরণ|সেট\\s*[A-Zক-হ]?\\s*=", p):\n        return "reasoning"\n    if re.search(r"বয়স", p) and digits >= 2:\n        return "reasoning"\n    if re.search(r"কাজ", p) and re.search(r"দিন|ঘণ্টা|ঘণ্টায়|জন", p) and digits >= 2:\n        return "reasoning"\n    if re.search(r"ত্রিভুজ|বৃত্ত|ব্যাসার্ধ|কর্ণ|সমবাহু|সমকোণ|বাহু|পঞ্চভুজ|ষড়ভুজ|বহুভুজ|চতুর্ভুজ", p) and (digits >= 1 or "মান" in p or "সমষ্টি" in p):\n        return "reasoning"\n    if re.search(r"অঙ্ক", p) and digits >= 1:\n        return "reasoning"\n    if re.search(r"ঘড়ি", p) and re.search(r"সেকেন্ড|মিনিট", p):\n        return "reasoning"\n    if re.search(r"ট্রেন", p) and re.search(r"স্টেশন", p):\n        return "reasoning"\n    if re.search(r"রূপান্তর", p):\n        return "reasoning"\n    if re.search(r"গুণ", p) and re.search(r"বাড়বে|কমবে|কত\\s*গুণ", p):\n        return "reasoning"\n    if re.search(r"\\u221a|\\u00d7|\\u03c0", p):\n        return "reasoning"\n    if re.search(r"(?<![0-9\\u09e6-\\u09ef])[<>\\u2264\\u2265](?![0-9\\u09e6-\\u09ef]*%)", p):\n        return "reasoning"\n    if re.search(r"\\b[0-9\\u09e6-\\u09ef]+\\s*/\\s*[0-9\\u09e6-\\u09ef]+\\b", p):\n        return "reasoning"\n    if _CALC_RE.search(p) and not has_ctx:\n        return "reasoning"\n    return "fast"\n\ndef _meta_block(row, kind):\n    # task_type reflects the ACTUAL processing tier so the metadata never contradicts the\n    # prompt style (e.g. a work-rate word-problem handled in the reasoning tier is labelled\n    # Reasoning-Calculation even when the coarse rule-based tag said Generative-QA).\n    task_type = {"reasoning": "Reasoning-Calculation", "meaning": "Definition-Meaning",\n                 "goldref": "Factual-QA"}.get(kind, str(row.get(\'task_type\', \'\')))\n    parts = [\n        f"domain={row.get(\'domain\',\'\')}", f"subdomain={row.get(\'subdomain\',\'\')}",\n        f"task_type={task_type}", f"complexity={row.get(\'complexity_level\',\'\')}",\n        f"language_mix={row.get(\'language_mix\',\'\')}", f"is_mcq={bool(row.get(\'is_mcq\'))}",\n        f"has_negation={bool(row.get(\'has_negation\'))}",\n        f"has_number_or_date={bool(row.get(\'has_number_or_date\'))}",\n    ]\n    return ("Metadata: " + " | ".join(parts) + \'\\n\' +\n            "(Consider these signals -- domain, task type, complexity, negation, and whether "\n            "numeric/date content is involved -- before analyzing the Question and Response below, "\n            "then decide.)" + \'\\n\\n\')\n\ndef build_messages(row, kind):\n    q = str(row.get("prompt_bn", "")); a = str(row.get("response_bn", ""))\n    has_ctx = bool(row.get("has_context"))\n    ctx = str(row.get("context", "[NULL]"))\n    gold = str(row.get("gold_hint", "")).strip()\n    gold_line = (f"A verified correct answer to this question (from a trusted "\n                 f"knowledge base) is: \\"{gold}\\". Use it as strong evidence and judge the "\n                 f"Response STRICTLY against it: output 1 (FAITHFUL) ONLY if the Response conveys "\n                 f"this same answer (exact match OR an unambiguous paraphrase/synonym); output 0 "\n                 f"(HALLUCINATION) if it gives a different, conflicting, or contradictory answer. "\n                 f"Do not accept a Response that disagrees with this verified answer, even if it "\n                 f"sounds confident.\\n") if gold else ""\n    rag = str(row.get("rag_evidence", "")).strip()\n    rag_block = (f"Reference material (retrieved; may be incomplete or unrelated -- use it ONLY "\n                 f"if it clearly bears on the Question, otherwise rely on your own knowledge):\\n"\n                 f"{rag}\\n\\n") if rag else ""\n    meta_block = _meta_block(row, kind)\n\n    if kind == "reasoning":\n        ctx_block = f"Context: {ctx}\\n\\n" if has_ctx else ""\n        content = (\n            "You are a math expert. Verify this Bengali word-problem answer with precision, "\n            "but write CONCISELY.\\n\\n"\n            f"{ctx_block}{rag_block}{meta_block}Question: {q}\\nResponse: {a}\\n\\n{gold_line}"\n            "In <think></think>: pick the right method "\n            "(work-rate: 1/A+1/B+...=1/together; average: total/count; ratio: solve "\n            "proportionally; SI=P*R*T/100; ages/LCM/GCD: set up the equation), compute, then "\n            "VERIFY before concluding. Only key equations/numbers -- no restating the problem.\\n"\n            "After </think>, output exactly one line:\\n"\n            "[LABEL]: 1  (Response matches your verified answer) or [LABEL]: 0  (Response is wrong)"\n        )\n        return [{"role": "user", "content": content}]\n\n    if kind == "meaning":\n        is_grammar    = bool(re.search(r"সন্ধি|সমাস|ব্যাসবাক্য|উপসর্গ|প্রত্যয়|বিচ্ছেদ", q))\n        is_figurative = bool(re.search(r"বাগধারা|প্রবাদ|ভাবার্থ", q))\n\n        meaning_calib = ("Be STRICT: unless the Response clearly conveys the SAME meaning, "\n                         "choose 0. A vague, incomplete, or slightly-different meaning "\n                         "is a hallucination (0)." if CALIBRATION == "strict" else "")\n\n        if is_grammar:\n            grammar_hint = str(row.get("grammar_hint", "")).strip()\n            hint_line = (f"A trusted grammar reference gives the correct answer as: \\"{grammar_hint}\\". "\n                        f"Treat this as ground truth.\\n\\n") if grammar_hint else ""\n            content = (\n                "You are an expert in Bengali (বাংলা) grammar: সন্ধি (sandhi/euphonic combination), "\n                "সমাস (compound-word classification), উপসর্গ/প্রত্যয় (prefixes/suffixes).\\n\\n"\n                "Worked example (format only -- NOT the question to judge):\\n"\n                "Question: \'রাজপুত্র\' শব্দের সন্ধি বিচ্ছেদ কী?\\n"\n                "Response: রাজ + পুত্র\\n"\n                "Check: \'রাজপুত্র\' = রাজ + পুত্র (স্বরসন্ধি)। Response সঠিক।\\n"\n                "[LABEL]: 1\\n\\n"\n                "Now judge the ACTUAL question:\\n"\n                f"{rag_block}{meta_block}Question: {q}\\nResponse: {a}\\n\\n{hint_line}{gold_line}"\n                "Work out the correct grammatical breakdown/classification step by step, then compare "\n                "it EXACTLY (the specific term or breakdown, not just \'same general topic\') against "\n                "the Response. A response about the right topic but the WRONG specific term or "\n                "breakdown is still a hallucination (0).\\n"\n                f"{meaning_calib}\\n"\n                "Answer in 2-3 short sentences ONLY -- no markdown headers, no bold (**text**), "\n                "no restating the question, no preamble like \'Let\'s analyze\'. State the correct "\n                "term/breakdown and compare it directly against the Response.\\n"\n                "For উপসর্গ class, these name the SAME class -- treat as equal, not a mismatch: "\n                "\'বাংলা\' = \'খাঁটি বাংলা\' = \'খাঁটি বাংলা উপসর্গ\'; \'তৎসম\' = \'সংস্কৃত\'; "\n                "\'বিদেশি\' = \'বিদেশী\'. Judge on the class itself, not on the short/long wording.\\n"\n                "For সমাস, the trailing word \'সমাস\' and hyphens are OPTIONAL -- treat these as "\n                "the same: \'দ্বন্দ্ব\' = \'দ্বন্দ্ব সমাস\'; \'সপ্তমী তৎপুরুষ\' = \'সপ্তমী-তৎপুরুষ\' "\n                "= \'সপ্তমী তৎপুরুষ সমাস\'. Match on the সমাস type, not the exact wording.\\n"\n                "Output your check, then the final line EXACTLY:\\n"\n                "[LABEL]: 1  (exact match) or [LABEL]: 0  (wrong term/breakdown)"\n            )\n            return [{"role": "user", "content": content}]\n\n        if is_figurative:\n            idiom = str(row.get("idiom_hint", "")).strip()\n            idiom_line = (f"A trusted idiom dictionary gives the correct meaning as: \\"{idiom}\\". "\n                          f"Treat this as the ground-truth meaning.\\n\\n") if idiom else ""\n            content = (\n                "You are an expert in Bengali idioms (বাগধারা) and proverbs.\\n\\n"\n                "Two worked examples (format only -- NOT the question to judge):\\n"\n                "Question: \\"গোঁফ খেজুরে\\" এর ভাবার্থ কী?\\n"\n                "Response: অত্যন্ত অলস ব্যক্তি\\n"\n                "Meaning: \\"গোঁফ খেজুরে\\" means an extremely lazy person; the Response conveys the same idea.\\n"\n                "[LABEL]: 1\\n\\n"\n                "Question: \\"গোঁফ খেজুরে\\" এর ভাবার্থ কী?\\n"\n                "Response: খুব সাহসী মানুষ\\n"\n                "Meaning: it means a lazy person, NOT a brave one -- the Response states a different meaning.\\n"\n                "[LABEL]: 0\\n\\n"\n                "Now judge the ACTUAL question:\\n"\n                f"{rag_block}{meta_block}Question: {q}\\nResponse: {a}\\n\\n{idiom_line}{gold_line}"\n                "Step 1: state the correct FIGURATIVE meaning of the phrase in ONE short sentence "\n                "(not a literal word-by-word translation).\\n"\n                "Step 2: does the Response convey that SAME meaning? A paraphrase or synonym counts as "\n                "CORRECT (1). But if the Response gives a DIFFERENT meaning, or only the literal sense, "\n                "it is a hallucination (0) -- do NOT accept a wrong meaning just because it is "\n                "confidently written.\\n"\n                f"{meaning_calib}\\n"\n                "DECISION RULE: compare the Response to the meaning YOU stated in Step 1. If they express the same core idea -- a synonym, a paraphrase, or different wording all count as a match -- output 1. Output 0 ONLY if the Response gives a genuinely different, opposite, or wrong meaning; do NOT output 0 merely for different wording or slightly less detail.\\n"\n                "Output your Step-1 meaning sentence, then the final line EXACTLY:\\n"\n                "[LABEL]: 1  (matches) or [LABEL]: 0  (different/wrong meaning)"\n            )\n            return [{"role": "user", "content": content}]\n\n        literal_hint = str(row.get("literal_hint", "")).strip()\n        fig_hint = str(row.get("idiom_hint", "")).strip()\n        sense_ref = ""\n        if literal_hint:\n            sense_ref += f"A dictionary reference gives the literal meaning as: \\"{literal_hint}\\".\\n"\n        if fig_hint:\n            sense_ref += (f"This word also has an established figurative/symbolic meaning: "\n                         f"\\"{fig_hint}\\".\\n")\n        if sense_ref:\n            if fig_hint:\n                sense_ref += ("Treat both as trusted references. For an idiom or set expression, BOTH "\n                             "the literal and the established figurative meaning are acceptable answers "\n                             "-- mark the Response CORRECT (1) if it conveys EITHER of them (exact match "\n                             "or a reasonable paraphrase/synonym), even when the question says শাব্দিক. "\n                             "Mark it 0 ONLY if the Response gives a meaning that matches NEITHER sense "\n                             "(a genuinely different or wrong meaning).\\n\\n")\n            else:\n                sense_ref += ("Treat this as a trusted reference -- the Response is correct if it "\n                             "matches this meaning (exact match or paraphrase).\\n\\n")\n        content = (\n            "You are an expert Bengali lexicographer judging word meanings. A question phrased as "\n            "\\"শাব্দিক অর্থ\\" (literal meaning) usually wants the literal/dictionary sense, but some "\n            "words are conventionally used in an established figurative or symbolic sense instead -- "\n            "judge the Response against whichever sense is actually correct for THIS word.\\n\\n"\n            "Worked example (format only -- NOT the question to judge):\\n"\n            "Question: \\"ফ্ল্যাট\\" এর শাব্দিক অর্থ কী?\\n"\n            "Response: চ্যাপ্টা বা সমতল\\n"\n            "Check: এটি সাধারণ শব্দের আক্ষরিক/অভিধানগত অর্থ জিজ্ঞাসা। \'চ্যাপ্টা বা সমতল\' সঠিক আক্ষরিক অর্থ।\\n"\n            "[LABEL]: 1\\n\\n"\n            "Now judge the ACTUAL question:\\n"\n            f"{rag_block}{meta_block}Question: {q}\\nResponse: {a}\\n\\n{sense_ref}{gold_line}"\n            "Step 1: state the correct meaning in ONE short sentence -- literal, or the established "\n            "figurative/symbolic sense if that is what this word conventionally means.\\n"\n            "Step 2: does the Response convey that SAME meaning (literal OR figurative, whichever is "\n            "correct for this word)? A synonym or paraphrase counts as correct (1).\\n"\n            f"{meaning_calib}\\n"\n            "DECISION RULE: compare the Response to the meaning YOU stated in Step 1. If they express the same core idea -- a synonym, a paraphrase, or different wording all count as a match -- output 1. Output 0 ONLY if the Response gives a genuinely different, opposite, or wrong meaning; do NOT output 0 merely for different wording or slightly less detail.\\n"\n            "Output your Step-1 meaning sentence, then the final line EXACTLY:\\n"\n            "[LABEL]: 1  (matches) or [LABEL]: 0  (different/wrong meaning)"\n        )\n        return [{"role": "user", "content": content}]\n\n    at = _AT_HINT.get(infer_answer_type(q), "")\n    hint = (" " + at) if at else ""\n    if has_ctx:\n        ctx_block = f"Context: {ctx}\\n\\n"\n        instr = ("Is the Response SUPPORTED BY the Context above (different wording is fine)? "\n                 "Flag it (0) only if it contradicts the Context or adds facts not found in it.")\n    else:\n        ctx_block = ""\n        instr = ("Using your knowledge, is the Response factually CORRECT? A wrong fact, date, "\n                 "name, place or number is a hallucination (0).")\n    if CALIBRATION == "strict":\n        calib = (" Do not assume correctness -- verify first. Choose 0 if wrong/fabricated/unsupported; "\n                 "choose 1 only when confirmed correct.")\n    elif CALIBRATION == "lean_faithful":\n        calib = (" Choose 0 only if genuinely confident it is wrong/unsupported; correct, plausible, "\n                 "or paraphrased responses get 1 -- do not flag for brevity or different wording.")\n    else:\n        calib = ""\n    content = (\n        "You are an expert Bengali fact-checker.\\n\\n"\n        f"{ctx_block}{rag_block}Question: {q}\\nResponse: {a}\\n\\n{gold_line}{instr}{hint}{calib}\\n"\n        "Work out the correct answer yourself in 1-2 short sentences, then on the FINAL line "\n        "output EXACTLY:\\n"\n        "[LABEL]: 1 (faithful) or [LABEL]: 0 (hallucinated)."\n    )\n    return [{"role": "user", "content": content}]\n\ndef parse_verdict(text):\n    t = unicodedata.normalize("NFC", text)\n    m = re.findall(r"\\[(?:LABEL|FACT|FAITHFUL|VERDICT|LABELED)\\]\\s*[:：]?\\s*([01])", t)\n    if m: return int(m[-1])\n    lines = [l.strip() for l in t.strip().splitlines() if l.strip()]\n    if lines:\n        mm = re.search(r"\\b([01])\\b", lines[-1])\n        if mm: return int(mm.group(1))\n    tl = t.lower()\n    EN = ["hallucinat", "not faithful", "fabricat", "incorrect", "unsupported", "wrong",\n          "does not match", "does not accurately", "incorrectly"]\n    BN = [unicodedata.normalize("NFC", w) for w in ["সত্য নয়", "ভুল", "সঠিক নয়", "ঠিক নয়", "অসত্য"]]\n    if any(w in tl for w in EN) or any(w in t for w in BN): return 0\n    return 1\n\ndef _txt_tok(tok):\n    return getattr(tok, "tokenizer", tok)\n\ndef load_model(model_name, retries=2):\n    for attempt in range(retries + 1):\n        gc.collect(); torch.cuda.empty_cache()\n        print(f"\\nLoading {model_name} ... (attempt {attempt+1}/{retries+1})")\n        try:\n            model, tokenizer = FastLanguageModel.from_pretrained(\n                model_name=model_name, max_seq_length=MAX_SEQ_LENGTH, dtype=None,\n                load_in_4bit=True, device_map={"": 0}, use_exact_model_name=True)\n        except RuntimeError as e:\n            if "out of memory" not in str(e).lower() or attempt == retries: raise\n            print("OOM while loading -- cleanup + retry in 5s...")\n            for _ in range(3): gc.collect()\n            torch.cuda.empty_cache(); torch.cuda.synchronize(); torch.cuda.ipc_collect()\n            time.sleep(5); continue\n        FastLanguageModel.for_inference(model)\n        _tt = _txt_tok(tokenizer)\n        _tt.padding_side = "left"\n        if _tt.pad_token is None: _tt.pad_token = _tt.eos_token\n        print(f"Loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")\n        return model, tokenizer\n\ndef clear_model(model, tokenizer):\n    try:\n        from accelerate.hooks import remove_hook_from_module\n        remove_hook_from_module(model, recurse=True)\n    except Exception: pass\n    del model, tokenizer\n    for _ in range(3): gc.collect()\n    torch.cuda.empty_cache(); torch.cuda.synchronize(); torch.cuda.ipc_collect()\n\nclass StopOnLabel(StoppingCriteria):\n    """Stop once EVERY sequence in the batch has emitted a parseable [LABEL] verdict.\n    Checked only every CHECK_EVERY steps, and only on the last ~24 tokens of the generated\n    region (not the whole sequence). The previous version re-decoded the ENTIRE sequence on\n    every token once any \']\' appeared -- an O(n^2) cost plus a GPU->CPU sync per step that\n    silently dominated wall-clock on the long reasoning-tier generations."""\n    CHECK_EVERY = 4\n    def __init__(self, tokenizer, input_len, batch_size):\n        self.tokenizer = tokenizer; self.input_len = input_len\n        self.done = [False]*batch_size; self.step = 0\n    def __call__(self, input_ids, scores, **kwargs):\n        self.step += 1\n        if self.step % self.CHECK_EVERY != 0:\n            return False\n        for j in range(input_ids.shape[0]):\n            if not self.done[j]:\n                tail = self.tokenizer.decode(input_ids[j, self.input_len:][-24:], skip_special_tokens=True)\n                if re.search(r"\\[LABEL\\]\\s*[:：]?\\s*[01]", tail): self.done[j] = True\n        return all(self.done)\n\ndef _to_chat_text(tokenizer, messages, thinking):\n    try:\n        return tokenizer.apply_chat_template(messages, tokenize=False,\n                                             add_generation_prompt=True, enable_thinking=thinking)\n    except TypeError:\n        try: return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n        except Exception: return messages[0]["content"]\n    except Exception:\n        return messages[0]["content"]\n\ndef load_cache(path):\n    """Load a tier\'s on-disk checkpoint, if any. Returns {} if the file doesn\'t exist yet."""\n    if not os.path.exists(path):\n        return {}\n    with open(path, "r", encoding="utf-8") as f:\n        return {item["idx"]: (item["verdict"], item["reasoning"]) for item in json.load(f)}\n\ndef append_cache(path, new_results):\n    """Merge freshly-generated rows into the on-disk checkpoint. Called after EVERY batch (not\n    just at the end) so a Kaggle disconnect / kernel restart only loses the one in-flight batch --\n    rerunning the cell resumes exactly where it left off instead of redoing the whole tier."""\n    existing = []\n    if os.path.exists(path):\n        with open(path, "r", encoding="utf-8") as f:\n            existing = json.load(f)\n    seen = {item["idx"] for item in existing}\n    for idx, (v, r) in new_results.items():\n        if idx not in seen:\n            existing.append({"idx": int(idx), "verdict": v, "reasoning": r})\n    with open(path, "w", encoding="utf-8") as f:\n        json.dump(existing, f, ensure_ascii=False, indent=2)\n\ndef _gen_or_split(model, tokenizer, batch_texts, batch_idx, gen_kwargs):\n    """Generate one batch; on CUDA OOM, clear the cache and split the batch in half, recursing\n    until it fits (worst case 1 row at a time). This makes the run STRUCTURALLY OOM-proof: batch\n    sizing no longer has to be predicted perfectly (impossible to do reliably -- free VRAM shifts\n    with fragmentation over hundreds of batches), because any batch that doesn\'t fit self-heals\n    instead of crashing the kernel. A single row that STILL OOMs at max_len gets a fallback verdict\n    (with a diagnostic memory dump -- if THIS ever fires, it is allocator FRAGMENTATION, not a\n    genuinely huge row: a single ~2k-token row cannot exhaust a T4 on its own).\n\n    pad_to_multiple_of=64 buckets every batch\'s padded length to a fixed grid (64/128/192/...)\n    instead of an arbitrary exact length. Dynamic (token-budget) batching produces a DIFFERENT\n    padded shape almost every batch -- each odd shape forces PyTorch\'s caching allocator to carve\n    a brand-new block, and over hundreds of batches the free-list fragments until even a small\n    single-row allocation can\'t find a contiguous slot (this is the "OOM\'d even alone" symptom,\n    where a batch that should easily fit fails anyway). Bucketing means many batches share the\n    same handful of shapes, so freed blocks actually get reused instead of endlessly fragmenting.\n    """\n    torch.cuda.empty_cache()\n    tt = _txt_tok(tokenizer)\n    try:\n        enc = tt(batch_texts, return_tensors="pt", padding=True, truncation=True,\n                        max_length=MAX_INPUT_LEN, pad_to_multiple_of=64).to(DEVICE)\n        input_len = enc["input_ids"].shape[1]\n        stopper = StoppingCriteriaList([StopOnLabel(tt, input_len, len(batch_texts))])\n        with torch.no_grad():\n            out = model.generate(**enc, **gen_kwargs, use_cache=True,\n                                 pad_token_id=tt.pad_token_id, stopping_criteria=stopper)\n        res = {}\n        for seq, idx in zip(out, batch_idx):\n            gen = tt.decode(seq[input_len:], skip_special_tokens=True).strip()\n            res[idx] = (parse_verdict(gen), gen)\n        del enc, out\n        return res\n    except RuntimeError as e:\n        if "out of memory" not in str(e).lower():\n            raise\n        try: del enc\n        except NameError: pass\n        try: del out\n        except NameError: pass\n        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()\n        if len(batch_texts) == 1:\n            alloc = torch.cuda.memory_allocated() / 1e9; reserv = torch.cuda.memory_reserved() / 1e9\n            print(f"    [OOM] row {batch_idx[0]} OOM\'d even alone (allocated={alloc:.2f}GB "\n                  f"reserved={reserv:.2f}GB) -- assigning fallback verdict")\n            return {batch_idx[0]: (parse_verdict(""), "")}\n        mid = len(batch_texts) // 2\n        print(f"    [OOM] batch of {len(batch_texts)} did not fit -> retry as {mid}+{len(batch_texts)-mid}")\n        res = {}\n        res.update(_gen_or_split(model, tokenizer, batch_texts[:mid], batch_idx[:mid], gen_kwargs))\n        res.update(_gen_or_split(model, tokenizer, batch_texts[mid:], batch_idx[mid:], gen_kwargs))\n        return res\n\ndef run_inference(model, tokenizer, df_subset, kind, label, batch_size=None, cache_path=None):\n    gen_kwargs = GEN_CONFIG[kind]; thinking = (kind in REASONING_KINDS)\n    bs = batch_size or BATCH_SIZE\n    results = load_cache(cache_path) if cache_path else {}\n    todo = df_subset[~df_subset.index.isin(results.keys())]\n    if cache_path and len(todo) < len(df_subset):\n        print(f"  [{label}] resuming from {cache_path}: {len(df_subset)-len(todo)} already done, "\n              f"{len(todo)} remaining")\n    if len(todo) == 0:\n        return results\n    rows_all = todo.to_dict("records"); idx_all = todo.index.tolist()\n\n    _tt = _txt_tok(tokenizer)\n    texts_all = [_to_chat_text(tokenizer, build_messages(r, kind), thinking) for r in rows_all]\n    lens_all = [len(_tt(t)["input_ids"]) for t in texts_all]\n    order = sorted(range(len(rows_all)), key=lambda k: lens_all[k])\n    texts = [texts_all[k] for k in order]; indices = [idx_all[k] for k in order]\n    lens = [lens_all[k] for k in order]\n\n    max_new = int(gen_kwargs.get("max_new_tokens", 128))\n    eff = [l + max_new for l in lens]\n    packed = []\n    cur_idx, cur_txt, cur_max = [], [], 0\n    for i in range(len(texts)):\n        cand_max = max(cur_max, eff[i])\n        if cur_idx and ((len(cur_idx) + 1) * cand_max > SEQ_TOKEN_BUDGET or len(cur_idx) >= bs):\n            packed.append((cur_idx, cur_txt))\n            cur_idx, cur_txt, cur_max = [], [], 0\n            cand_max = eff[i]\n        cur_idx.append(indices[i]); cur_txt.append(texts[i]); cur_max = cand_max\n    if cur_idx:\n        packed.append((cur_idx, cur_txt))\n\n    for bi, (batch_idx, batch_texts) in enumerate(tqdm(packed, desc=label)):\n        batch_results = _gen_or_split(model, tokenizer, batch_texts, batch_idx, gen_kwargs)\n        results.update(batch_results)\n        if cache_path:\n            append_cache(cache_path, batch_results)\n        gc.collect(); torch.cuda.empty_cache()\n        if bi > 0 and bi % 50 == 0:\n            print(f"    [mem check @ batch {bi}] allocated={torch.cuda.memory_allocated()/1e9:.2f}GB "\n                  f"reserved={torch.cuda.memory_reserved()/1e9:.2f}GB")\n    return results\n\ndef main():\n    df = pd.read_parquet(args.shard)\n    df = df.set_index("__idx__")\n    print(f"[{args.tag}] shard rows: {len(df)}  tiers: {df[\'tier\'].value_counts().to_dict()}", flush=True)\n\n    model, tokenizer = load_model(MODEL_NAME)\n    print(f"[{args.tag}] VRAM after load: allocated={torch.cuda.memory_allocated()/1e9:.2f}GB "\n          f"reserved={torch.cuda.memory_reserved()/1e9:.2f}GB", flush=True)\n\n    t0 = time.time()\n    results = {}\n    for kind, bs, cache_name in [\n        ("fast",      BATCH_SIZE,         f"cache_fast_{args.tag}.json"),\n        ("meaning",   BATCH_SIZE_MEANING, f"cache_meaning_{args.tag}.json"),\n        ("reasoning", BATCH_SIZE_MATH,    f"cache_reasoning_{args.tag}.json"),\n    ]:\n        sub = df[df["tier"] == kind]\n        tt0 = time.time()\n        results.update(run_inference(model, tokenizer, sub, kind, f"{args.tag}:{kind}",\n                                     batch_size=bs, cache_path=cache_name))\n        print(f"[{args.tag}] {kind}: {len(sub)} rows done in {(time.time()-tt0)/60:.1f} min", flush=True)\n\n    def _reasoning_empty(text):\n        t = re.sub(r"</?think>", "", str(text))\n        pre = re.split(r"\\[LABEL\\]", t)[0]\n        return len(re.sub(r"\\s", "", pre)) < 10\n\n    def _overwrite_cache(path, new_results):\n        existing = []\n        if os.path.exists(path):\n            with open(path, "r", encoding="utf-8") as f:\n                existing = json.load(f)\n        upd = {int(k): v for k, v in new_results.items()}\n        for item in existing:\n            if item["idx"] in upd:\n                v, r = upd.pop(item["idx"])\n                item["verdict"], item["reasoning"] = v, r\n        for idx, (v, r) in upd.items():\n            existing.append({"idx": int(idx), "verdict": v, "reasoning": r})\n        tmp = path + ".tmp"\n        with open(tmp, "w", encoding="utf-8") as f:\n            json.dump(existing, f, ensure_ascii=False, indent=2)\n        os.replace(tmp, path)\n\n    class _MinGateStop(StoppingCriteria):\n        CHECK_EVERY = 4\n        def __init__(self, tok, input_len, bsz, min_new):\n            self.tok = tok; self.input_len = input_len; self.min_new = min_new\n            self.done = [False]*bsz; self.step = 0\n        def __call__(self, input_ids, scores, **kw):\n            self.step += 1\n            if self.step % self.CHECK_EVERY:\n                return False\n            if input_ids.shape[1] - self.input_len < self.min_new:\n                return False\n            for j in range(input_ids.shape[0]):\n                if not self.done[j]:\n                    tail = self.tok.decode(input_ids[j, self.input_len:][-24:], skip_special_tokens=True)\n                    if re.search(r"\\[LABEL\\]\\s*[:：]?\\s*[01]", tail): self.done[j] = True\n            return all(self.done)\n\n    _redo = [i for i in df[df["tier"] == "reasoning"].index\n             if i in results and _reasoning_empty(results[i][1])]\n    if _redo:\n        print(f"[{args.tag}] math re-verify: {len(_redo)} empty-reasoning rows -- redoing batched "\n              f"(model still loaded, forced computation)", flush=True)\n        _FORCE = ("\\n\\nIMPORTANT: You MUST show the full step-by-step calculation with the actual "\n                  "numbers in <think></think> BEFORE the final line. Do NOT output [LABEL] until you "\n                  "have computed and verified the answer.")\n        tt = _txt_tok(tokenizer)\n        fixed = 0\n        RV_BATCH = 4\n        RV_BUDGET_MIN = 20  # hard wall-clock cap: re-verify is optional refinement,\n                            # never let it hang the whole run (one stuck worker => dead run)\n        _rv_t0 = time.time()\n        def _rv_generate(chunk):\n            texts = []\n            for ri in chunk:\n                msgs = build_messages(df.loc[ri], "reasoning")\n                msgs[-1]["content"] = msgs[-1]["content"] + _FORCE\n                texts.append(_to_chat_text(tokenizer, msgs, True))\n            enc = tt(texts, return_tensors="pt", padding=True, truncation=True,\n                     max_length=MAX_INPUT_LEN, pad_to_multiple_of=64).to(DEVICE)\n            stopper = StoppingCriteriaList([_MinGateStop(tt, enc["input_ids"].shape[1], len(chunk), 72)])\n            with torch.no_grad():\n                out = model.generate(**enc, max_new_tokens=350, min_new_tokens=72, do_sample=False,\n                                     pad_token_id=tt.pad_token_id, stopping_criteria=stopper,\n                                     use_cache=True)\n            res = {}\n            for seq, ri in zip(out, chunk):\n                gen = tt.decode(seq[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()\n                res[ri] = gen\n            del enc, out\n            return res\n        _chunks = [_redo[s:s+RV_BATCH] for s in range(0, len(_redo), RV_BATCH)]\n        for chunk in tqdm(_chunks, desc=f"{args.tag}:math-reverify"):\n            if (time.time() - _rv_t0) / 60 > RV_BUDGET_MIN:\n                print(f"[{args.tag}] math re-verify: {RV_BUDGET_MIN}-min budget reached -- "\n                      f"stopping early, keeping original verdicts for the rest", flush=True)\n                break\n            try:\n                gens = _rv_generate(chunk)\n            except RuntimeError as e:\n                if "out of memory" not in str(e).lower():\n                    raise\n                gc.collect(); torch.cuda.empty_cache()\n                gens = {}\n                for ri in chunk:\n                    try:\n                        gens.update(_rv_generate([ri]))\n                    except RuntimeError:\n                        gc.collect(); torch.cuda.empty_cache()\n            upd = {}\n            for ri, gen in gens.items():\n                if (not _reasoning_empty(gen)) and ("[LABEL]" in gen):\n                    results[ri] = (parse_verdict(gen), gen + "\\n[math-reverify]")\n                    upd[ri] = results[ri]\n                    fixed += 1\n            if upd:\n                _overwrite_cache(f"cache_reasoning_{args.tag}.json", upd)\n            gc.collect(); torch.cuda.empty_cache()\n        print(f"[{args.tag}] math re-verify: {len(_redo)} rows done in "\n              f"{(time.time()-_rv_t0)/60:.1f} min ({fixed} fixed, rest kept original)", flush=True)\n\n    clear_model(model, tokenizer)\n    print(f"[{args.tag}] ALL TIERS DONE in {(time.time()-t0)/60:.1f} min", flush=True)\n\nif __name__ == "__main__":\n    main()\n'
with open(WORKER_PATH, "w", encoding="utf-8") as _f:
    _f.write(_worker_code)
print(f"Wrote {WORKER_PATH} ({len(_worker_code)} chars)")


## 7. Decide Verdicts (dual-GPU launch + merge, speed-first LLM)

Assigns every row's tier, splits the rows into two workload-balanced shards, launches the two GPU
worker subprocesses, polls until both finish, then merges their per-tier checkpoint files into
`results`. Everything downstream (assembly loop, `reasoning.txt`, `rag_retrieval.txt`,
`submission.csv`) is unchanged.


### Shared inference helpers (outer scope)

`load_cache` is used by the merge/resume logic on the **dual-GPU path**, and `load_model`/`run_inference`/`clear_model` by the **single-GPU fallback path**. They are defined here at notebook scope (the dual-GPU *worker subprocess* carries its own identical copy inside `dual_gpu_worker.py`).

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList
def _txt_tok(tok):
    return getattr(tok, "tokenizer", tok)

def load_model(model_name, retries=2):
    for attempt in range(retries + 1):
        gc.collect(); torch.cuda.empty_cache()
        print(f"\nLoading {model_name} ... (attempt {attempt+1}/{retries+1})")
        try:
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name=model_name, max_seq_length=MAX_SEQ_LENGTH, dtype=None,
                load_in_4bit=True, device_map={"": 0}, use_exact_model_name=True)
        except RuntimeError as e:
            if "out of memory" not in str(e).lower() or attempt == retries: raise
            print("OOM while loading -- cleanup + retry in 5s...")
            for _ in range(3): gc.collect()
            torch.cuda.empty_cache(); torch.cuda.synchronize(); torch.cuda.ipc_collect()
            time.sleep(5); continue
        FastLanguageModel.for_inference(model)
        _tt = _txt_tok(tokenizer)
        _tt.padding_side = "left"
        if _tt.pad_token is None: _tt.pad_token = _tt.eos_token
        print(f"Loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        return model, tokenizer

def clear_model(model, tokenizer):
    try:
        from accelerate.hooks import remove_hook_from_module
        remove_hook_from_module(model, recurse=True)
    except Exception: pass
    del model, tokenizer
    for _ in range(3): gc.collect()
    torch.cuda.empty_cache(); torch.cuda.synchronize(); torch.cuda.ipc_collect()

class StopOnLabel(StoppingCriteria):
    """Stop once EVERY sequence in the batch has emitted a parseable [LABEL] verdict.
    Checked only every CHECK_EVERY steps, and only on the last ~24 tokens of the generated
    region (not the whole sequence). The previous version re-decoded the ENTIRE sequence on
    every token once any ']' appeared -- an O(n^2) cost plus a GPU->CPU sync per step that
    silently dominated wall-clock on the long reasoning-tier generations."""
    CHECK_EVERY = 4
    def __init__(self, tokenizer, input_len, batch_size):
        self.tokenizer = tokenizer; self.input_len = input_len
        self.done = [False]*batch_size; self.step = 0
    def __call__(self, input_ids, scores, **kwargs):
        self.step += 1
        if self.step % self.CHECK_EVERY != 0:
            return False
        for j in range(input_ids.shape[0]):
            if not self.done[j]:
                tail = self.tokenizer.decode(input_ids[j, self.input_len:][-24:], skip_special_tokens=True)
                if re.search(r"\[LABEL\]\s*[:：]?\s*[01]", tail): self.done[j] = True
        return all(self.done)

def _to_chat_text(tokenizer, messages, thinking):
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False,
                                             add_generation_prompt=True, enable_thinking=thinking)
    except TypeError:
        try: return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception: return messages[0]["content"]
    except Exception:
        return messages[0]["content"]

def load_cache(path):
    """Load a tier's on-disk checkpoint, if any. Returns {} if the file doesn't exist yet."""
    if not os.path.exists(path):
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return {item["idx"]: (item["verdict"], item["reasoning"]) for item in json.load(f)}

def append_cache(path, new_results):
    """Merge freshly-generated rows into the on-disk checkpoint. Called after EVERY batch (not
    just at the end) so a Kaggle disconnect / kernel restart only loses the one in-flight batch --
    rerunning the cell resumes exactly where it left off instead of redoing the whole tier."""
    existing = []
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            existing = json.load(f)
    seen = {item["idx"] for item in existing}
    for idx, (v, r) in new_results.items():
        if idx not in seen:
            existing.append({"idx": int(idx), "verdict": v, "reasoning": r})
    with open(path, "w", encoding="utf-8") as f:
        json.dump(existing, f, ensure_ascii=False, indent=2)

def _gen_or_split(model, tokenizer, batch_texts, batch_idx, gen_kwargs):
    """Generate one batch; on CUDA OOM, clear the cache and split the batch in half, recursing
    until it fits (worst case 1 row at a time). This makes the run STRUCTURALLY OOM-proof: batch
    sizing no longer has to be predicted perfectly (impossible to do reliably -- free VRAM shifts
    with fragmentation over hundreds of batches), because any batch that doesn't fit self-heals
    instead of crashing the kernel. A single row that STILL OOMs at max_len gets a fallback verdict
    (with a diagnostic memory dump -- if THIS ever fires, it is allocator FRAGMENTATION, not a
    genuinely huge row: a single ~2k-token row cannot exhaust a T4 on its own).

    pad_to_multiple_of=64 buckets every batch's padded length to a fixed grid (64/128/192/...)
    instead of an arbitrary exact length. Dynamic (token-budget) batching produces a DIFFERENT
    padded shape almost every batch -- each odd shape forces PyTorch's caching allocator to carve
    a brand-new block, and over hundreds of batches the free-list fragments until even a small
    single-row allocation can't find a contiguous slot (this is the "OOM'd even alone" symptom,
    where a batch that should easily fit fails anyway). Bucketing means many batches share the
    same handful of shapes, so freed blocks actually get reused instead of endlessly fragmenting.
    """
    torch.cuda.empty_cache()
    tt = _txt_tok(tokenizer)
    try:
        enc = tt(batch_texts, return_tensors="pt", padding=True, truncation=True,
                        max_length=MAX_INPUT_LEN, pad_to_multiple_of=64).to(DEVICE)
        input_len = enc["input_ids"].shape[1]
        stopper = StoppingCriteriaList([StopOnLabel(tt, input_len, len(batch_texts))])
        with torch.no_grad():
            out = model.generate(**enc, **gen_kwargs, use_cache=True,
                                 pad_token_id=tt.pad_token_id, stopping_criteria=stopper)
        res = {}
        for seq, idx in zip(out, batch_idx):
            gen = tt.decode(seq[input_len:], skip_special_tokens=True).strip()
            res[idx] = (parse_verdict(gen), gen)
        del enc, out
        return res
    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise
        try: del enc
        except NameError: pass
        try: del out
        except NameError: pass
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        if len(batch_texts) == 1:
            alloc = torch.cuda.memory_allocated() / 1e9; reserv = torch.cuda.memory_reserved() / 1e9
            print(f"    [OOM] row {batch_idx[0]} OOM'd even alone (allocated={alloc:.2f}GB "
                  f"reserved={reserv:.2f}GB) -- assigning fallback verdict")
            return {batch_idx[0]: (parse_verdict(""), "")}
        mid = len(batch_texts) // 2
        print(f"    [OOM] batch of {len(batch_texts)} did not fit -> retry as {mid}+{len(batch_texts)-mid}")
        res = {}
        res.update(_gen_or_split(model, tokenizer, batch_texts[:mid], batch_idx[:mid], gen_kwargs))
        res.update(_gen_or_split(model, tokenizer, batch_texts[mid:], batch_idx[mid:], gen_kwargs))
        return res

def run_inference(model, tokenizer, df_subset, kind, label, batch_size=None, cache_path=None):
    gen_kwargs = GEN_CONFIG[kind]; thinking = (kind in REASONING_KINDS)
    bs = batch_size or BATCH_SIZE
    results = load_cache(cache_path) if cache_path else {}
    todo = df_subset[~df_subset.index.isin(results.keys())]
    if cache_path and len(todo) < len(df_subset):
        print(f"  [{label}] resuming from {cache_path}: {len(df_subset)-len(todo)} already done, "
              f"{len(todo)} remaining")
    if len(todo) == 0:
        return results
    rows_all = todo.to_dict("records"); idx_all = todo.index.tolist()

    _tt = _txt_tok(tokenizer)
    texts_all = [_to_chat_text(tokenizer, build_messages(r, kind), thinking) for r in rows_all]
    lens_all = [len(_tt(t)["input_ids"]) for t in texts_all]
    order = sorted(range(len(rows_all)), key=lambda k: lens_all[k])
    texts = [texts_all[k] for k in order]; indices = [idx_all[k] for k in order]
    lens = [lens_all[k] for k in order]

    max_new = int(gen_kwargs.get("max_new_tokens", 128))
    eff = [l + max_new for l in lens]
    packed = []
    cur_idx, cur_txt, cur_max = [], [], 0
    for i in range(len(texts)):
        cand_max = max(cur_max, eff[i])
        if cur_idx and ((len(cur_idx) + 1) * cand_max > SEQ_TOKEN_BUDGET or len(cur_idx) >= bs):
            packed.append((cur_idx, cur_txt))
            cur_idx, cur_txt, cur_max = [], [], 0
            cand_max = eff[i]
        cur_idx.append(indices[i]); cur_txt.append(texts[i]); cur_max = cand_max
    if cur_idx:
        packed.append((cur_idx, cur_txt))

    for bi, (batch_idx, batch_texts) in enumerate(tqdm(packed, desc=label)):
        batch_results = _gen_or_split(model, tokenizer, batch_texts, batch_idx, gen_kwargs)
        results.update(batch_results)
        if cache_path:
            append_cache(cache_path, batch_results)
        gc.collect(); torch.cuda.empty_cache()
        if bi > 0 and bi % 50 == 0:
            print(f"    [mem check @ batch {bi}] allocated={torch.cuda.memory_allocated()/1e9:.2f}GB "
                  f"reserved={torch.cuda.memory_reserved()/1e9:.2f}GB")
    return results

In [ ]:
import subprocess, sys, time

df_llm = df_test

def _pick_tier(r):
    return route_tier(r["prompt_bn"], bool(r["has_context"]))

df_llm = df_llm.copy()
df_llm["tier"] = df_llm.apply(_pick_tier, axis=1)
print(f"fast: {(df_llm['tier']=='fast').sum()}   "
      f"meaning: {(df_llm['tier']=='meaning').sum()}   reasoning: {(df_llm['tier']=='reasoning').sum()}   "
      f"total: {len(df_test)}")

def _plen(row):
    return (len(str(row.get("context", ""))) + len(str(row.get("prompt_bn", "")))
            + len(str(row.get("response_bn", ""))) + len(str(row.get("rag_evidence", ""))))

shard_tag = {}
for tier_name, sub in df_llm.groupby("tier"):
    lens = {idx: _plen(sub.loc[idx]) for idx in sub.index}
    order = sorted(sub.index, key=lambda i: lens[i], reverse=True)
    for k, idx in enumerate(order):
        shard_tag[idx] = "gpu0" if k % 2 == 0 else "gpu1"

df_llm["_shard"] = df_llm.index.map(shard_tag)
df_shard0 = df_llm[df_llm["_shard"] == "gpu0"].reset_index().rename(columns={df_llm.index.name or "index": "__idx__"})
df_shard1 = df_llm[df_llm["_shard"] == "gpu1"].reset_index().rename(columns={df_llm.index.name or "index": "__idx__"})
print(f"shard gpu0: {len(df_shard0)} rows   shard gpu1: {len(df_shard1)} rows")

def _n_remaining(df_subset, cache_path):
    cached = load_cache(cache_path)
    return len(df_subset) - sum(1 for idx in df_subset.index if idx in cached)

if len(df_llm) == 0:
    results = {}
else:
    n_gpus = torch.cuda.device_count()
    if n_gpus < 2:
        print(f"Only {n_gpus} GPU visible (dual-GPU needs 2 -- check Settings -> Accelerator is "
              f"'GPU T4 x2') -- falling back to single-process inference on GPU 0.")
        CACHE_FAST = "cache_fast.json"
        CACHE_MEANING, CACHE_REASONING = "cache_meaning.json", "cache_reasoning.json"
        df_fast    = df_llm[df_llm["tier"] == "fast"]
        df_meaning = df_llm[df_llm["tier"] == "meaning"]
        df_math    = df_llm[df_llm["tier"] == "reasoning"]
        n_remaining = (_n_remaining(df_fast, CACHE_FAST)
                       + _n_remaining(df_meaning, CACHE_MEANING) + _n_remaining(df_math, CACHE_REASONING))
        if n_remaining == 0:
            results = {}
            for p in [CACHE_FAST, CACHE_MEANING, CACHE_REASONING]:
                results.update(load_cache(p))
        else:
            t0 = time.time()
            model, tokenizer = load_model(MODEL_NAME)
            results = {}
            results.update(run_inference(model, tokenizer, df_fast, "fast", "fast",
                                         batch_size=BATCH_SIZE, cache_path=CACHE_FAST))
            results.update(run_inference(model, tokenizer, df_meaning, "meaning", "meaning",
                                         batch_size=BATCH_SIZE_MEANING, cache_path=CACHE_MEANING))
            results.update(run_inference(model, tokenizer, df_math, "reasoning", "reasoning",
                                         batch_size=BATCH_SIZE_MATH, cache_path=CACHE_REASONING))
            clear_model(model, tokenizer)
            print(f"LLM wall-clock: {(time.time()-t0)/60:.1f} min for {len(df_llm)} samples")
    else:
        SHARD0_PATH, SHARD1_PATH, CONFIG_PATH = "shard_gpu0.parquet", "shard_gpu1.parquet", "dual_gpu_config.json"
        df_shard0.to_parquet(SHARD0_PATH)
        df_shard1.to_parquet(SHARD1_PATH)
        json.dump({
            "MODEL_NAME": MODEL_NAME, "MAX_SEQ_LENGTH": MAX_SEQ_LENGTH, "MAX_INPUT_LEN": MAX_INPUT_LEN,
            "SEQ_TOKEN_BUDGET": SEQ_TOKEN_BUDGET, "BATCH_SIZE": BATCH_SIZE,
            "BATCH_SIZE_MEANING": BATCH_SIZE_MEANING, "BATCH_SIZE_MATH": BATCH_SIZE_MATH,
            "CALIBRATION": CALIBRATION, "SEED": SEED, "GEN_CONFIG": GEN_CONFIG,
            "REASONING_KINDS": list(REASONING_KINDS),
        }, open(CONFIG_PATH, "w", encoding="utf-8"), ensure_ascii=False)

        def _launch(shard_path, tag, gpu_id, logfile):
            env = dict(os.environ); env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
            log = open(logfile, "w", encoding="utf-8")
            return subprocess.Popen([sys.executable, WORKER_PATH, "--shard", shard_path,
                                     "--config", CONFIG_PATH, "--tag", tag],
                                    env=env, stdout=log, stderr=subprocess.STDOUT)

        already_done = all(
            _n_remaining(df_shard0.set_index("__idx__")[df_shard0.set_index("__idx__")["tier"] == t],
                        f"cache_{t}_gpu0.json") == 0
            and _n_remaining(df_shard1.set_index("__idx__")[df_shard1.set_index("__idx__")["tier"] == t],
                             f"cache_{t}_gpu1.json") == 0
            for t in ["fast", "meaning", "reasoning"]
        )
        if already_done:
            print("All tiers already fully cached on both shards -- skipping subprocess launch entirely.")
        else:
            def _prog(df_shard, tag):
                dfi = df_shard.set_index("__idx__")
                parts, done = [], 0
                for tier in ["fast", "meaning", "reasoning"]:
                    idxs = dfi.index[dfi["tier"] == tier]
                    try:
                        cached = load_cache(f"cache_{tier}_{tag}.json")
                        d = sum(1 for i in idxs if i in cached)
                    except Exception:
                        d = -1
                    if d >= 0: done += d
                    parts.append(f"{tier[:4]} {d if d>=0 else '?'}/{len(idxs)}")
                return "  ".join(parts), done, len(df_shard)
            t0 = time.time()
            p0 = _launch(SHARD0_PATH, "gpu0", 0, "gpu0.log")
            p1 = _launch(SHARD1_PATH, "gpu1", 1, "gpu1.log")
            print(f"Launched worker subprocesses: gpu0 pid={p0.pid}, gpu1 pid={p1.pid}. Polling...")
            GRACE_MIN = 30  # once one worker exits, do not wait forever for the other (guards a
                            # hung re-verify / CUDA stall) -- all tiers are cached by then.
            grace_deadline = None
            while p0.poll() is None or p1.poll() is None:
                time.sleep(20)
                st0 = "running" if p0.poll() is None else f"done rc={p0.returncode}"
                st1 = "running" if p1.poll() is None else f"done rc={p1.returncode}"
                pr0, d0, n0 = _prog(df_shard0, "gpu0")
                pr1, d1, n1 = _prog(df_shard1, "gpu1")
                el = (time.time() - t0) / 60
                print(f"  [{el:5.1f} min] gpu0 ({st0}) {d0}/{n0}:  {pr0}")
                print(f"               gpu1 ({st1}) {d1}/{n1}:  {pr1}")
                if (p0.poll() is not None) or (p1.poll() is not None):
                    if grace_deadline is None:
                        grace_deadline = time.time() + GRACE_MIN * 60
                        print(f"  (one worker done -- waiting up to {GRACE_MIN} min for the "
                              f"other, then proceeding with cached results)")
                    elif time.time() > grace_deadline:
                        for _p, _tag in [(p0, "gpu0"), (p1, "gpu1")]:
                            if _p.poll() is None:
                                print(f"!!! {_tag} still running {GRACE_MIN} min after the other "
                                      f"finished -- terminating (all tiers cached; skipping any "
                                      f"unfinished re-verify refinement) !!!")
                                _p.terminate()
                                try: _p.wait(timeout=30)
                                except Exception: _p.kill()
                        break
            print(f"Both workers finished. Total wall-clock: {(time.time()-t0)/60:.1f} min")
            if p0.returncode != 0 or p1.returncode != 0:
                print("!!! one or both workers exited with a non-zero code -- check gpu0.log / gpu1.log !!!")

        results = {}
        for tag in ["gpu0", "gpu1"]:
            for tier in ["fast", "meaning", "reasoning"]:
                results.update(load_cache(f"cache_{tier}_{tag}.json"))
        print(f"Merged results from both GPUs: {len(results)}/{len(df_llm)} rows")

        missing = [idx for idx in df_llm.index if idx not in results]
        if missing:
            by_shard_tier = {}
            for idx in missing:
                tier = df_llm.loc[idx, "tier"]
                tag = "gpu0" if idx in set(df_shard0["__idx__"]) else "gpu1"
                by_shard_tier[(tag, tier)] = by_shard_tier.get((tag, tier), 0) + 1
            print(f"!!! {len(missing)}/{len(df_llm)} rows missing after merge -- a worker likely "
                  f"died partway (check gpu0.log / gpu1.log). Breakdown:")
            for (tag, tier), n in sorted(by_shard_tier.items()):
                print(f"    {tag} / {tier}: {n} rows missing")
            print("    Assigning fallback verdicts so submission.csv is still complete -- "
                  "RERUN this cell to resume: cached rows are skipped, only the gap gets redone.")
            for idx in missing:
                results[idx] = (parse_verdict(""), "[worker-crash fallback: no result generated]")

# ---- Math re-verification pass: redo empty-reasoning rows with forced computation ----
def _reasoning_empty(text):
    # "empty" = model bailed with essentially NO work before [LABEL]. Measured by CONTENT
    # LENGTH, not digit-presence: real reasoning can be verbal or symbolic (algebra x(x+y),
    # image/no-problem answers) with no digits and must NOT be re-run. True bails have ~0 chars
    # before [LABEL] (empty </think>); real reasoning is 15+ chars -- clean separation in data.
    t = re.sub(r"</?think>", "", str(text))
    pre = re.split(r"\[LABEL\]", t)[0]
    return len(re.sub(r"\s", "", pre)) < 10

_reverify = [idx for idx in df_llm.index
             if df_llm.loc[idx, "tier"] == "reasoning"
             and idx in results and _reasoning_empty(results[idx][1])]
_worker_retried = any("[math-reverify]" in str(v[1]) for v in results.values())
print(f"Math re-verification: {len(_reverify)} empty-reasoning rows remain"
      + (" (workers already retried in-process -- these resisted forcing; skipping main-process redo)"
         if _worker_retried else ""))
if _reverify and not _worker_retried:
    _FORCE = ("\n\nIMPORTANT: You MUST show the full step-by-step calculation with the actual "
              "numbers in <think></think> BEFORE the final line. Do NOT output [LABEL] until you "
              "have computed and verified the answer.")
    _rvfixed = 0
    try:
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize(); torch.cuda.ipc_collect()
        _rvm, _rvt = load_model(MODEL_NAME)
        _rvtt = _txt_tok(_rvt)
        for _rn, _ri in enumerate(_reverify, 1):
            print(f"  [math-reverify fallback] {_rn}/{len(_reverify)}", flush=True)
            _row = df_test.loc[_ri]
            _msgs = build_messages(_row, "reasoning")
            _msgs[-1]["content"] = _msgs[-1]["content"] + _FORCE
            _text = _to_chat_text(_rvt, _msgs, True)
            _enc = _rvtt([_text], return_tensors="pt", truncation=True,
                         max_length=MAX_INPUT_LEN).to(DEVICE)
            with torch.no_grad():
                _out = _rvm.generate(**_enc, max_new_tokens=350, min_new_tokens=72,
                                     do_sample=False, pad_token_id=_rvtt.pad_token_id)
            _gen = _rvtt.decode(_out[0][_enc["input_ids"].shape[1]:],
                                skip_special_tokens=True).strip()
            if (not _reasoning_empty(_gen)) and ("[LABEL]" in _gen):
                results[_ri] = (parse_verdict(_gen), _gen + "\n[math-reverify]")
                _rvfixed += 1
        clear_model(_rvm, _rvt)
    except Exception as _e:
        print(f"  [math-reverify] skipped due to error (originals kept): {_e}")
    print(f"Math re-verification: {_rvfixed}/{len(_reverify)} rows now show computation and were updated")

# ---- Deterministic self-consistency override: model's OWN final computed number vs response ----
def _bn2en(s):
    return str(s).translate(str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789"))

def _extract_final_equation_number(pre_text):
    t = _bn2en(pre_text)
    t = re.sub(r"</?think>|\[/?think\]|\[math-reverify\]", "", t).strip()
    parts = [p.strip() for p in re.split(r"[.\n]", t) if p.strip()]
    if not parts:
        return None
    last = parts[-1]
    m = re.search(r"=\s*(-?\d+\.?\d*)\s*$", last)
    if not m:
        if last.lower().strip() == "verified" and len(parts) >= 2:
            m = re.search(r"=\s*(-?\d+\.?\d*)\s*$", parts[-2])
        if not m:
            return None
    return float(m.group(1))

def _resp_number(s):
    s2 = _bn2en(s).replace(",", "")
    m = re.match(r"\s*(\d+)\s*\(\s*(\d+)\s*/\s*(\d+)\s*\)\s*%?", s2)
    if m:
        whole, num, den = map(int, m.groups())
        return whole + num / den
    ns = re.findall(r"-?\d+\.?\d*", s2)
    return float(ns[-1]) if ns else None

_n_selfcheck = 0
for _sci in df_llm.index:
    if df_llm.loc[_sci, "tier"] != "reasoning" or _sci not in results:
        continue
    _v, _r = results[_sci]
    if int(_v) != 1:
        continue
    if "[LABEL]" not in _r:
        continue  # truncated (no verdict tag) -- text is incomplete scratch-work, not a
                  # trustworthy final answer; skip rather than risk reading a mid-derivation
                  # number (e.g. a restated GIVEN value) as if it were the model's conclusion
    _pre = re.split(r"\[LABEL\]", _r)[0]
    _fn = _extract_final_equation_number(_pre)
    if _fn is None:
        continue
    _rn = _resp_number(df_test.loc[_sci, "response_bn"])
    if _rn is None:
        continue
    if abs(_fn - _rn) > max(0.05, 0.02 * max(abs(_fn), abs(_rn))):
        results[_sci] = (0, _r + f"\n[self-check-override: model computed {_fn}, response states {_rn}]")
        _n_selfcheck += 1

# ---- Deterministic spelling override (dictionary lookup, single-word বানান questions only) ----
_n_spell = 0
if SPELL_DICT:
    for _spi in df_llm.index:
        if _spi not in results:
            continue
        _sq = unicodedata.normalize('NFC', str(df_test.loc[_spi, 'prompt_bn']))
        if ('বানান' not in _sq) or ('বিপরীত' in _sq) or ('বাক্য' in _sq):
            continue
        _sws = re.findall(r'[\u0980-\u09ff]{2,}', unicodedata.normalize('NFC', str(df_test.loc[_spi, 'response_bn'])))
        if len(_sws) != 1:
            continue
        _sw = _sws[0]
        _wrong_q = bool(re.search(r'ভুল\s*বানান|অশুদ্ধ\s*বানান', _sq))
        _indict = _sw in SPELL_DICT
        _sv = (0 if _indict else 1) if _wrong_q else (1 if _indict else 0)
        _v0, _r0 = results[_spi]
        if int(_v0) != _sv:
            results[_spi] = (_sv, _r0 + f"\n[spelling-override: '{_sw}' in_dict={_indict} wrong_q={_wrong_q} -> {_sv}]")
            _n_spell += 1

final_preds, reasonings, src, tiers = [], [], [], []
for idx in df_test.index:
    row = df_test.loc[idx]
    v, r = results[idx]; v = int(v)
    final_preds.append(v); reasonings.append(r); src.append("llm"); tiers.append(_pick_tier(row))

print(f"Predictions: {len(final_preds)}   0: {final_preds.count(0)}   1: {final_preds.count(1)}")


## 7b. Export `reasoning.txt` (diagnostic -- inspect for truncation, tune token size)

One plain-text file with every row's raw model output, tagged by tier and flagged if it ran
out of tokens before reaching `[LABEL]` (no tag found = truncated mid-`<think>`). Use this to
check whether `GEN_CONFIG`'s token budgets (fast/meaning/reasoning) are actually enough --
raise a tier's `max_new_tokens` only if this shows real truncations there, not by guessing.

In [ ]:
n_trunc = 0
with open("reasoning.txt", "w", encoding="utf-8") as f:
    for pos, idx in enumerate(df_test.index):
        row = df_test.loc[idx]
        r = reasonings[pos]
        truncated = (tiers[pos] != "gold") and ("[LABEL]" not in r)
        n_trunc += int(truncated)
        line = f"id={row['id']}  tier={tiers[pos]}  task_type={row['task_type']}  pred={final_preds[pos]}"
        if truncated:
            line += "  [TRUNCATED -- no [LABEL] tag, ran out of tokens for this tier]"
        f.write("=" * 100 + "\n")
        f.write(line + "\n")
        f.write(f"Q: {row['prompt_bn']}\n")
        f.write(f"A: {row['response_bn']}\n")
        f.write("Reasoning:\n" + r + "\n")

print(f"Wrote reasoning.txt ({len(df_test)} rows).")
print(f"Truncated (no [LABEL] found -- ran out of tokens): {n_trunc}")
if n_trunc:
    from collections import Counter
    tc = Counter(tiers[i] for i in range(len(reasonings)) if tiers[i] != "gold" and "[LABEL]" not in reasonings[i])
    print("  truncated by tier:", dict(tc))
    print("  -> if a tier shows real truncations, raise ITS max_new_tokens in GEN_CONFIG (cell 3).")

## 7c. Export `rag_retrieval.txt` (diagnostic -- inspect RAG performance)

For every NO-CONTEXT question: whether the RAG step found evidence above `RAG_RERANK_MIN`,
the actual retrieved passage(s), and the final verdict.

In [ ]:
n_noctx = 0; n_evidence = 0

with open("rag_retrieval.txt", "w", encoding="utf-8") as f:
    for pos, idx in enumerate(df_test.index):
        row = df_test.loc[idx]
        if bool(row.get("has_context")):
            continue
        n_noctx += 1
        ev = str(row.get("rag_evidence", "")).strip()
        has_ev = bool(ev)
        n_evidence += int(has_ev)
        line = f"id={row['id']}  tier={tiers[pos]}  pred={final_preds[pos]}"
        line += "  evidence_found=" + ("YES" if has_ev else "NO")
        f.write("=" * 100 + "\n")
        f.write(line + "\n")
        f.write(f"Q: {row['prompt_bn']}\n")
        f.write(f"A: {row['response_bn']}\n")
        f.write("Retrieved evidence:\n" +
               (ev if has_ev else "(none -- reranker found nothing above RAG_RERANK_MIN)") + "\n")

print(f"Wrote rag_retrieval.txt ({n_noctx} no-context rows).")
print(f"  evidence found: {n_evidence}/{n_noctx} ({n_evidence/max(1,n_noctx)*100:.1f}%)")

## 8. Write submission.csv

In [ ]:
sub = pd.DataFrame({"id": df_test["id"].values, "label": [int(p) for p in final_preds]})
sub.to_csv("submission.csv", index=False)
print(f"Wrote submission.csv  ({len(sub)} rows)")
print(sub.head())